# Evaluating the Effect of 2-Opt Local Search Intensity in a Hybrid Genetic Algorithm for the Traveling Salesman Problem

## Experimental Implementation

This notebook implements and evaluates a standard Genetic Algorithm (GA) and hybrid GA with varying levels of 2-opt local search intensity for solving benchmark Traveling Salesman Problem (TSP) instances.

### Experimental Configurations
- Standard GA: 0% 2-opt intensity
- Low intensity: 10%
- Moderate intensity: 30%
- High intensity: 50%

### TSPLIB Benchmark Instances
- eil51
- eil76
- kroA100
- ch150
- kroA200
- pr264

In [ ]:
import os
import sys
import platform

print("Python version:", sys.version.split()[0])
print("CPU cores:", os.cpu_count())
print("Platform:", platform.platform())

In [ ]:
import numpy as np
import pandas as pd
import random
import time
import math
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Libraries loaded successfully.")

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

INSTANCE_NAMES = [
    "eil51",
    "eil76",
    "kroA100",
    "ch150",
    "kroA200",
    "pr264"
]

OPTIMAL_TOUR_LENGTHS = {
    "eil51": 426,
    "eil76": 538,
    "kroA100": 21282,
    "ch150": 6528,
    "kroA200": 29368,
    "pr264": 49135
}

POPULATION_SIZE = 100
MAX_GENERATIONS = 500

CROSSOVER_RATE = 0.80
MUTATION_RATE = 0.20

TOURNAMENT_SIZE = 3
ELITE_COUNT = 1

TWO_OPT_INTERVAL = 10

TWO_OPT_INTENSITIES = {
    "GA": 0.00,
    "GA_2OPT_10": 0.10,
    "GA_2OPT_30": 0.30,
    "GA_2OPT_50": 0.50
}

print("Experiment configuration loaded.")

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/mastqe/tsplib/master/eil51.tsp"
filename = "eil51.tsp"

urllib.request.urlretrieve(url, filename)

print(f"{filename} downloaded successfully.")

In [ ]:
with open("eil51.tsp", "r") as file:
    content = file.read()

print(content[:1500])

In [ ]:
def load_tsplib_coordinates(filename):
    coordinates = {}
    reading_coordinates = False

    with open(filename, "r") as file:
        for line in file:
            line = line.strip()

            if line == "NODE_COORD_SECTION":
                reading_coordinates = True
                continue

            if line == "EOF":
                break

            if reading_coordinates:
                parts = line.split()

                city_id = int(parts[0])
                x = float(parts[1])
                y = float(parts[2])

                coordinates[city_id] = (x, y)

    return coordinates


eil51_coords = load_tsplib_coordinates("eil51.tsp")

print("Number of cities:", len(eil51_coords))
print("\nFirst 5 cities:")

for city_id in list(eil51_coords.keys())[:5]:
    print(city_id, eil51_coords[city_id])

In [ ]:
# ==========================================
# STEP 7: TSPLIB EUC_2D DISTANCE
# ==========================================

def euc_2d_distance(coord1, coord2):
    x1, y1 = coord1
    x2, y2 = coord2

    distance = math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    # TSPLIB nearest-integer convention
    return int(distance + 0.5)


# Test using cities 1 and 2
test_distance = euc_2d_distance(
    eil51_coords[1],
    eil51_coords[2]
)

print("City 1 coordinates:", eil51_coords[1])
print("City 2 coordinates:", eil51_coords[2])
print("Distance between cities 1 and 2:", test_distance)

In [ ]:
def build_distance_matrix(coordinates):
    city_ids = sorted(coordinates.keys())
    n = len(city_ids)

    matrix = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i, j] = euc_2d_distance(
                    coordinates[city_ids[i]],
                    coordinates[city_ids[j]]
                )

    return matrix


eil51_distance_matrix = build_distance_matrix(eil51_coords)

print("Distance matrix shape:", eil51_distance_matrix.shape)

print("\nFirst 5 x 5 section:")
print(eil51_distance_matrix[:5, :5])

print("\nSymmetric:",
      np.array_equal(eil51_distance_matrix,
                     eil51_distance_matrix.T))

print("Diagonal all zero:",
      np.all(np.diag(eil51_distance_matrix) == 0))

In [ ]:
# ==========================================
# STEP 9: VISUALIZE EIL51
# ==========================================

x_coords = [eil51_coords[i][0] for i in sorted(eil51_coords)]
y_coords = [eil51_coords[i][1] for i in sorted(eil51_coords)]

plt.figure(figsize=(8, 6))
plt.scatter(x_coords, y_coords)

# Label each city
for city_id in sorted(eil51_coords):
    x, y = eil51_coords[city_id]
    plt.annotate(
        str(city_id),
        (x, y),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )

plt.title("TSPLIB eil51 – 51-City TSP Instance")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ==========================================
# STEP 10: TOUR REPRESENTATION AND LENGTH
# ==========================================

def calculate_tour_length(tour, distance_matrix):
    total_distance = 0

    # Distance between consecutive cities
    for i in range(len(tour) - 1):
        total_distance += distance_matrix[tour[i], tour[i + 1]]

    # Return from the final city to the starting city
    total_distance += distance_matrix[tour[-1], tour[0]]

    return total_distance


# Create a reproducible random tour for testing
rng = np.random.default_rng(42)

test_tour = rng.permutation(len(eil51_coords))

test_length = calculate_tour_length(
    test_tour,
    eil51_distance_matrix
)

print("Number of cities in tour:", len(test_tour))
print("Unique cities in tour:", len(set(test_tour)))
print("First 10 cities:", test_tour[:10])
print("Random tour length:", test_length)

In [ ]:
# ==========================================
# STEP 11: INITIALIZE GA POPULATION
# ==========================================

def initialize_population(population_size, num_cities, rng):
    population = []

    for _ in range(population_size):
        tour = rng.permutation(num_cities)
        population.append(tour)

    return population


# Reproducible test
test_rng = np.random.default_rng(42)

test_population = initialize_population(
    POPULATION_SIZE,
    len(eil51_coords),
    test_rng
)

print("Population size:", len(test_population))
print("Cities per tour:", len(test_population[0]))
print("Unique cities in first tour:", len(set(test_population[0])))

print("\nFirst tour:")
print(test_population[0])

print("\nFirst tour length:",
      calculate_tour_length(
          test_population[0],
          eil51_distance_matrix
      ))

In [ ]:
# ==========================================
# STEP 12: EVALUATE POPULATION
# ==========================================

def evaluate_population(population, distance_matrix):
    return np.array([
        calculate_tour_length(tour, distance_matrix)
        for tour in population
    ])


test_lengths = evaluate_population(
    test_population,
    eil51_distance_matrix
)

best_index = np.argmin(test_lengths)

print("Number of evaluated tours:", len(test_lengths))
print("Best tour length:", test_lengths[best_index])
print("Worst tour length:", np.max(test_lengths))
print("Mean tour length:", round(np.mean(test_lengths), 2))

In [ ]:
# ==========================================
# STEP 13: TOURNAMENT SELECTION
# ==========================================

def tournament_selection(population, tour_lengths, tournament_size, rng):
    # Randomly select candidate indices
    candidate_indices = rng.choice(
        len(population),
        size=tournament_size,
        replace=False
    )

    # Find the candidate with the shortest tour
    winner_index = candidate_indices[
        np.argmin(tour_lengths[candidate_indices])
    ]

    return population[winner_index].copy(), winner_index, candidate_indices


# Test tournament selection
test_rng = np.random.default_rng(42)

selected_parent, winner_index, candidates = tournament_selection(
    test_population,
    test_lengths,
    TOURNAMENT_SIZE,
    test_rng
)

print("Tournament candidates:", candidates)
print("Candidate tour lengths:", test_lengths[candidates])
print("Winner index:", winner_index)
print("Winner tour length:", test_lengths[winner_index])

print("\nSelected parent is valid:",
      len(selected_parent) == 51 and
      len(set(selected_parent)) == 51)

In [ ]:
# ==========================================
# STEP 14: ORDER CROSSOVER (OX)
# ==========================================

def order_crossover(parent1, parent2, rng):
    n = len(parent1)

    # Select two crossover points
    start, end = sorted(
        rng.choice(n, size=2, replace=False)
    )

    # Create empty children
    child1 = np.full(n, -1, dtype=int)
    child2 = np.full(n, -1, dtype=int)

    # Copy the selected segment
    child1[start:end + 1] = parent1[start:end + 1]
    child2[start:end + 1] = parent2[start:end + 1]

    def fill_remaining(child, donor, end):
        n = len(child)

        # Cities from donor that are not already in child
        remaining = [
            city for city in donor
            if city not in child
        ]

        # Positions are filled cyclically beginning after
        # the second crossover point
        positions = list(range(end + 1, n)) + list(range(0, end + 1))

        empty_positions = [
            pos for pos in positions
            if child[pos] == -1
        ]

        for pos, city in zip(empty_positions, remaining):
            child[pos] = city

        return child

    child1 = fill_remaining(child1, parent2, end)
    child2 = fill_remaining(child2, parent1, end)

    return child1, child2


# Test OX using two population members
test_rng = np.random.default_rng(42)

parent1 = test_population[0]
parent2 = test_population[1]

child1, child2 = order_crossover(
    parent1,
    parent2,
    test_rng
)

print("Child 1 cities:", len(child1))
print("Child 1 unique cities:", len(set(child1)))

print("Child 2 cities:", len(child2))
print("Child 2 unique cities:", len(set(child2)))

print("\nChild 1 valid:",
      len(child1) == 51 and len(set(child1)) == 51)

print("Child 2 valid:",
      len(child2) == 51 and len(set(child2)) == 51)

In [ ]:
# ==========================================
# STEP 15: INVERSION MUTATION
# ==========================================

def inversion_mutation(tour, rng):
    mutated_tour = tour.copy()
    n = len(mutated_tour)

    # Select two different positions
    start, end = sorted(
        rng.choice(n, size=2, replace=False)
    )

    # Reverse the selected segment
    mutated_tour[start:end + 1] = mutated_tour[start:end + 1][::-1]

    return mutated_tour, start, end


# Test inversion mutation
test_rng = np.random.default_rng(42)

original_tour = test_population[0].copy()

mutated_tour, start, end = inversion_mutation(
    original_tour,
    test_rng
)

print("Mutation positions:", start, "to", end)

print("\nOriginal segment:")
print(original_tour[start:end + 1])

print("\nReversed segment:")
print(mutated_tour[start:end + 1])

print("\nOriginal tour valid:",
      len(original_tour) == 51 and
      len(set(original_tour)) == 51)

print("Mutated tour valid:",
      len(mutated_tour) == 51 and
      len(set(mutated_tour)) == 51)

print("Tour changed:",
      not np.array_equal(original_tour, mutated_tour))

In [ ]:
# ==========================================
# STEP 16: STANDARD GENETIC ALGORITHM
# ==========================================

def run_standard_ga(
    distance_matrix,
    population_size=100,
    max_generations=500,
    crossover_rate=0.80,
    mutation_rate=0.20,
    tournament_size=3,
    elite_count=1,
    seed=42
):
    rng = np.random.default_rng(seed)
    num_cities = len(distance_matrix)

    # --------------------------------------
    # 1. Initialize population
    # --------------------------------------
    population = initialize_population(
        population_size,
        num_cities,
        rng
    )

    # Track convergence
    best_history = []

    # Track global best solution
    global_best_tour = None
    global_best_length = float("inf")

    # --------------------------------------
    # 2. Evolutionary loop
    # --------------------------------------
    for generation in range(max_generations):

        tour_lengths = evaluate_population(
            population,
            distance_matrix
        )

        # Sort population from best to worst
        sorted_indices = np.argsort(tour_lengths)

        # Best solution in current generation
        generation_best_index = sorted_indices[0]
        generation_best_length = tour_lengths[generation_best_index]

        # Update global best
        if generation_best_length < global_best_length:
            global_best_length = generation_best_length
            global_best_tour = population[generation_best_index].copy()

        best_history.append(global_best_length)

        # ----------------------------------
        # 3. Elitism
        # ----------------------------------
        new_population = []

        for i in range(elite_count):
            elite_index = sorted_indices[i]
            new_population.append(
                population[elite_index].copy()
            )

        # ----------------------------------
        # 4. Generate offspring
        # ----------------------------------
        while len(new_population) < population_size:

            parent1, _, _ = tournament_selection(
                population,
                tour_lengths,
                tournament_size,
                rng
            )

            parent2, _, _ = tournament_selection(
                population,
                tour_lengths,
                tournament_size,
                rng
            )

            # Crossover
            if rng.random() < crossover_rate:
                child1, child2 = order_crossover(
                    parent1,
                    parent2,
                    rng
                )
            else:
                child1 = parent1.copy()
                child2 = parent2.copy()

            # Mutation
            if rng.random() < mutation_rate:
                child1, _, _ = inversion_mutation(
                    child1,
                    rng
                )

            if rng.random() < mutation_rate:
                child2, _, _ = inversion_mutation(
                    child2,
                    rng
                )

            new_population.append(child1)

            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population

    # --------------------------------------
    # 5. Final evaluation
    # --------------------------------------
    final_lengths = evaluate_population(
        population,
        distance_matrix
    )

    final_best_index = np.argmin(final_lengths)

    if final_lengths[final_best_index] < global_best_length:
        global_best_length = final_lengths[final_best_index]
        global_best_tour = population[final_best_index].copy()

    return {
        "best_tour": global_best_tour,
        "best_length": int(global_best_length),
        "best_history": best_history
    }

In [ ]:
# ==========================================
# STEP 17: FIRST STANDARD GA TEST
# ==========================================

start_time = time.perf_counter()

ga_result = run_standard_ga(
    distance_matrix=eil51_distance_matrix,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    seed=42
)

elapsed_time = time.perf_counter() - start_time

best_length = ga_result["best_length"]

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

optimality_gap = (
    (best_length - optimal_length)
    / optimal_length
) * 100

print("Instance: eil51")
print("Algorithm: Standard GA")
print("Seed: 42")

print("\nBest tour length:", best_length)
print("Known optimal length:", optimal_length)
print("Optimality gap:", round(optimality_gap, 2), "%")
print("Execution time:", round(elapsed_time, 2), "seconds")

print("\nBest tour valid:",
      len(ga_result["best_tour"]) == 51 and
      len(set(ga_result["best_tour"])) == 51)

In [ ]:
# ==========================================
# STEP 18: STANDARD GA CONVERGENCE
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(ga_result["best_history"]) + 1),
    ga_result["best_history"]
)

plt.axhline(
    y=OPTIMAL_TOUR_LENGTHS["eil51"],
    linestyle="--",
    label="Known Optimum (426)"
)

plt.xlabel("Generation")
plt.ylabel("Best Tour Length")
plt.title("Standard GA Convergence on eil51")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# STEP 19: BEST-IMPROVEMENT 2-OPT
# ==========================================

def best_improvement_two_opt(tour, distance_matrix):
    """
    Apply best-improvement 2-opt until a 2-opt local optimum
    is reached.

    Returns:
        improved_tour
        improved_length
        num_moves
    """

    current_tour = tour.copy()
    n = len(current_tour)

    current_length = calculate_tour_length(
        current_tour,
        distance_matrix
    )

    num_moves = 0

    while True:

        best_delta = 0
        best_i = None
        best_j = None

        # Examine all eligible nonadjacent edge pairs
        for i in range(n - 1):

            a = current_tour[i]
            b = current_tour[(i + 1) % n]

            for j in range(i + 2, n):

                # Skip the pair consisting of the first and
                # last edges because they are adjacent in
                # the cyclic tour
                if i == 0 and j == n - 1:
                    continue

                c = current_tour[j]
                d = current_tour[(j + 1) % n]

                old_distance = (
                    distance_matrix[a, b] +
                    distance_matrix[c, d]
                )

                new_distance = (
                    distance_matrix[a, c] +
                    distance_matrix[b, d]
                )

                delta = new_distance - old_distance

                # Best improvement = largest reduction
                if delta < best_delta:
                    best_delta = delta
                    best_i = i
                    best_j = j

        # No improving 2-opt move remains
        if best_i is None:
            break

        # Reverse the segment between the selected edges
        current_tour[
            best_i + 1 : best_j + 1
        ] = current_tour[
            best_i + 1 : best_j + 1
        ][::-1]

        current_length += best_delta
        num_moves += 1

    # Recalculate once for verification
    verified_length = calculate_tour_length(
        current_tour,
        distance_matrix
    )

    assert current_length == verified_length

    return current_tour, int(verified_length), num_moves

In [ ]:
# ==========================================
# STEP 20: TEST 2-OPT ON GA BEST TOUR
# ==========================================

ga_best_tour = ga_result["best_tour"].copy()
ga_best_length = ga_result["best_length"]

start_time = time.perf_counter()

two_opt_tour, two_opt_length, two_opt_moves = (
    best_improvement_two_opt(
        ga_best_tour,
        eil51_distance_matrix
    )
)

two_opt_time = time.perf_counter() - start_time

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

before_gap = (
    (ga_best_length - optimal_length)
    / optimal_length
) * 100

after_gap = (
    (two_opt_length - optimal_length)
    / optimal_length
) * 100

print("Instance: eil51")

print("\nBefore 2-opt")
print("Tour length:", ga_best_length)
print("Optimality gap:", round(before_gap, 2), "%")

print("\nAfter best-improvement 2-opt")
print("Tour length:", two_opt_length)
print("Optimality gap:", round(after_gap, 2), "%")

print("\nImprovement in tour length:",
      ga_best_length - two_opt_length)

print("Number of accepted 2-opt moves:",
      two_opt_moves)

print("2-opt execution time:",
      round(two_opt_time, 4), "seconds")

print("\nImproved tour valid:",
      len(two_opt_tour) == 51 and
      len(set(two_opt_tour)) == 51)

print("2-opt never worsened solution:",
      two_opt_length <= ga_best_length)

In [ ]:
# ==========================================
# STEP 21: VERIFY 2-OPT LOCAL OPTIMUM
# ==========================================

def find_best_two_opt_move(tour, distance_matrix):
    n = len(tour)

    best_delta = 0
    best_i = None
    best_j = None

    for i in range(n - 1):

        a = tour[i]
        b = tour[(i + 1) % n]

        for j in range(i + 2, n):

            # First and last edges are adjacent
            # in a cyclic TSP tour
            if i == 0 and j == n - 1:
                continue

            c = tour[j]
            d = tour[(j + 1) % n]

            old_distance = (
                distance_matrix[a, b] +
                distance_matrix[c, d]
            )

            new_distance = (
                distance_matrix[a, c] +
                distance_matrix[b, d]
            )

            delta = new_distance - old_distance

            if delta < best_delta:
                best_delta = delta
                best_i = i
                best_j = j

    return best_delta, best_i, best_j


best_delta, best_i, best_j = find_best_two_opt_move(
    two_opt_tour,
    eil51_distance_matrix
)

print("Current tour length:", two_opt_length)
print("Best remaining 2-opt delta:", best_delta)
print("Best i:", best_i)
print("Best j:", best_j)

print("\n2-opt local optimum reached:",
      best_delta >= 0)

In [ ]:
# ==========================================
# STEP 22: HYBRID GA + 2-OPT
# ==========================================

def run_hybrid_ga(
    distance_matrix,
    two_opt_intensity,
    population_size=100,
    max_generations=500,
    crossover_rate=0.80,
    mutation_rate=0.20,
    tournament_size=3,
    elite_count=1,
    two_opt_interval=10,
    ga_seed=42,
    two_opt_seed=10042
):
    """
    Run the standard GA or hybrid GA + 2-opt.

    two_opt_intensity:
        0.00 = Standard GA
        0.10 = 10% of population
        0.30 = 30% of population
        0.50 = 50% of population
    """

    # Separate random generators
    # This is important for experimental control.
    ga_rng = np.random.default_rng(ga_seed)
    two_opt_rng = np.random.default_rng(two_opt_seed)

    num_cities = len(distance_matrix)

    # --------------------------------------
    # 1. Initialize population
    # --------------------------------------
    population = initialize_population(
        population_size,
        num_cities,
        ga_rng
    )

    best_history = []

    global_best_tour = None
    global_best_length = float("inf")

    total_two_opt_calls = 0
    total_two_opt_moves = 0

    # --------------------------------------
    # 2. Evolutionary loop
    # --------------------------------------
    for generation in range(max_generations):

        tour_lengths = evaluate_population(
            population,
            distance_matrix
        )

        sorted_indices = np.argsort(tour_lengths)

        generation_best_index = sorted_indices[0]
        generation_best_length = tour_lengths[
            generation_best_index
        ]

        if generation_best_length < global_best_length:
            global_best_length = generation_best_length
            global_best_tour = population[
                generation_best_index
            ].copy()

        # ----------------------------------
        # 3. Elitism
        # ----------------------------------
        new_population = []

        for i in range(elite_count):
            elite_index = sorted_indices[i]
            new_population.append(
                population[elite_index].copy()
            )

        # ----------------------------------
        # 4. Selection + crossover + mutation
        # ----------------------------------
        while len(new_population) < population_size:

            parent1, _, _ = tournament_selection(
                population,
                tour_lengths,
                tournament_size,
                ga_rng
            )

            parent2, _, _ = tournament_selection(
                population,
                tour_lengths,
                tournament_size,
                ga_rng
            )

            # Crossover
            if ga_rng.random() < crossover_rate:
                child1, child2 = order_crossover(
                    parent1,
                    parent2,
                    ga_rng
                )
            else:
                child1 = parent1.copy()
                child2 = parent2.copy()

            # Mutation
            if ga_rng.random() < mutation_rate:
                child1, _, _ = inversion_mutation(
                    child1,
                    ga_rng
                )

            if ga_rng.random() < mutation_rate:
                child2, _, _ = inversion_mutation(
                    child2,
                    ga_rng
                )

            new_population.append(child1)

            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population

        # ----------------------------------
        # 5. Scheduled 2-opt
        # ----------------------------------
        # generation + 1 makes the schedule:
        # 10, 20, 30, ..., 500
        if (
            two_opt_intensity > 0
            and (generation + 1) % two_opt_interval == 0
        ):

            num_two_opt = int(
                np.floor(
                    two_opt_intensity * population_size
                )
            )
            random_order = two_opt_rng.permutation(population_size
            )
            selected_indices = random_order[:num_two_opt]

            for idx in selected_indices:

                improved_tour, _, num_moves = (
                    best_improvement_two_opt(
                        population[idx],
                        distance_matrix
                    )
                )

                population[idx] = improved_tour

                total_two_opt_calls += 1
                total_two_opt_moves += num_moves

        # ----------------------------------
        # 6. Evaluate after local search
        # ----------------------------------
        updated_lengths = evaluate_population(
            population,
            distance_matrix
        )

        updated_best_index = np.argmin(updated_lengths)
        updated_best_length = updated_lengths[
            updated_best_index
        ]

        if updated_best_length < global_best_length:
            global_best_length = updated_best_length
            global_best_tour = population[
                updated_best_index
            ].copy()

        best_history.append(
            int(global_best_length)
        )

    return {
        "best_tour": global_best_tour,
        "best_length": int(global_best_length),
        "best_history": best_history,
        "two_opt_calls": total_two_opt_calls,
        "two_opt_moves": total_two_opt_moves
    }

In [ ]:
# ==========================================
# STEP 23: VALIDATE 0% CONFIGURATION
# ==========================================

control_result = run_hybrid_ga(
    distance_matrix=eil51_distance_matrix,
    two_opt_intensity=0.00,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    two_opt_interval=TWO_OPT_INTERVAL,
    ga_seed=42,
    two_opt_seed=10042
)

print("Original standard GA:", ga_result["best_length"])
print("Hybrid function at 0%:", control_result["best_length"])

print("\n2-opt calls:",
      control_result["two_opt_calls"])

print("Results match:",
      ga_result["best_length"]
      == control_result["best_length"])

print(
    "Convergence histories match:",
    ga_result["best_history"]
    == control_result["best_history"]
)

In [ ]:
# ==========================================
# STEP 23A: DIAGNOSE CONVERGENCE HISTORY
# ==========================================

standard_history = np.array(ga_result["best_history"])
control_history = np.array(control_result["best_history"])

print("Standard history length:", len(standard_history))
print("Control history length:", len(control_history))

print("\nFirst 10 - Standard GA:")
print(standard_history[:10])

print("\nFirst 10 - Hybrid 0%:")
print(control_history[:10])

# Check for one-generation shift
print(
    "\nStandard[1:] matches Control[:-1]:",
    np.array_equal(
        standard_history[1:],
        control_history[:-1]
    )
)

# Count differing positions
differences = np.where(
    standard_history != control_history
)[0]

print("Number of differing positions:", len(differences))

if len(differences) > 0:
    print("First differing generation index:", differences[0])

In [ ]:
# ==========================================
# STEP 24: CANONICAL ORDER CROSSOVER (OX)
# ==========================================

def order_crossover(parent1, parent2, rng):
    n = len(parent1)

    start, end = sorted(
        rng.choice(n, size=2, replace=False)
    )

    child1 = np.full(n, -1, dtype=int)
    child2 = np.full(n, -1, dtype=int)

    # Copy the segment between the crossover points
    child1[start:end + 1] = parent1[start:end + 1]
    child2[start:end + 1] = parent2[start:end + 1]

    def fill_remaining(child, donor, end):

        n = len(child)

        # Read donor cyclically beginning immediately
        # after the second crossover point
        donor_order = (
            list(donor[end + 1:]) +
            list(donor[:end + 1])
        )

        # Remove cities already copied into the child
        remaining = [
            city
            for city in donor_order
            if city not in child
        ]

        # Fill child cyclically beginning immediately
        # after the second crossover point
        positions = (
            list(range(end + 1, n)) +
            list(range(0, end + 1))
        )

        empty_positions = [
            pos
            for pos in positions
            if child[pos] == -1
        ]

        for pos, city in zip(
            empty_positions,
            remaining
        ):
            child[pos] = city

        return child

    child1 = fill_remaining(
        child1,
        parent2,
        end
    )

    child2 = fill_remaining(
        child2,
        parent1,
        end
    )

    return child1, child2

In [ ]:
# ==========================================
# STEP 24A: VALIDATE CANONICAL OX
# ==========================================

rng_test = np.random.default_rng(42)

parent1 = rng_test.permutation(51)
parent2 = rng_test.permutation(51)

child1, child2 = order_crossover(
    parent1,
    parent2,
    rng_test
)

print("Parent 1 valid:",
      len(parent1) == 51 and
      len(set(parent1)) == 51)

print("Parent 2 valid:",
      len(parent2) == 51 and
      len(set(parent2)) == 51)

print("Child 1 valid:",
      len(child1) == 51 and
      len(set(child1)) == 51)

print("Child 2 valid:",
      len(child2) == 51 and
      len(set(child2)) == 51)

print("\nChild 1 contains same cities as parent:",
      set(child1) == set(parent1))

print("Child 2 contains same cities as parent:",
      set(child2) == set(parent2))

In [ ]:
# ==========================================
# STEP 25: OFFICIAL 0% GA BASELINE
# ==========================================

import time

start_time = time.perf_counter()

baseline_result = run_hybrid_ga(
    distance_matrix=eil51_distance_matrix,
    two_opt_intensity=0.00,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    two_opt_interval=TWO_OPT_INTERVAL,
    ga_seed=42,
    two_opt_seed=10042
)

baseline_time = time.perf_counter() - start_time

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

baseline_gap = (
    (baseline_result["best_length"] - optimal_length)
    / optimal_length
) * 100

print("Instance: eil51")
print("Configuration: Standard GA (0% 2-opt)")
print("Seed: 42")

print("\nBest tour length:",
      baseline_result["best_length"])

print("Known optimal length:",
      optimal_length)

print("Optimality gap:",
      round(baseline_gap, 2), "%")

print("Execution time:",
      round(baseline_time, 2), "seconds")

print("\n2-opt calls:",
      baseline_result["two_opt_calls"])

print("2-opt moves:",
      baseline_result["two_opt_moves"])

print(
    "Best tour valid:",
    len(baseline_result["best_tour"]) == 51
    and len(set(baseline_result["best_tour"])) == 51
)

print(
    "History length:",
    len(baseline_result["best_history"])
)

In [ ]:
# ==========================================
# STEP 26: FIRST 2-OPT INTENSITY COMPARISON
# eil51 - Seed 42
# ==========================================

comparison_results = {}

# Keep our official baseline
comparison_results["GA"] = {
    "intensity": 0.00,
    "result": baseline_result,
    "runtime": baseline_time
}

# Run hybrid configurations
for config_name, intensity in [
    ("GA_2OPT_10", 0.10),
    ("GA_2OPT_30", 0.30),
    ("GA_2OPT_50", 0.50)
]:

    print(
        f"Running {config_name} "
        f"({intensity:.0%} intensity)..."
    )

    start_time = time.perf_counter()

    result = run_hybrid_ga(
        distance_matrix=eil51_distance_matrix,
        two_opt_intensity=intensity,
        population_size=POPULATION_SIZE,
        max_generations=MAX_GENERATIONS,
        crossover_rate=CROSSOVER_RATE,
        mutation_rate=MUTATION_RATE,
        tournament_size=TOURNAMENT_SIZE,
        elite_count=ELITE_COUNT,
        two_opt_interval=TWO_OPT_INTERVAL,
        ga_seed=42,
        two_opt_seed=10042
    )

    runtime = time.perf_counter() - start_time

    comparison_results[config_name] = {
        "intensity": intensity,
        "result": result,
        "runtime": runtime
    }

    print(
        "  Best length:",
        result["best_length"]
    )

    print(
        "  Runtime:",
        round(runtime, 2),
        "seconds"
    )

    print(
        "  2-opt calls:",
        result["two_opt_calls"]
    )

    print(
        "  2-opt moves:",
        result["two_opt_moves"]
    )

    print()

In [ ]:
# ==========================================
# STEP 27: SUMMARY TABLE
# ==========================================

summary_rows = []

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

for config_name, data in comparison_results.items():

    result = data["result"]
    intensity = data["intensity"]
    runtime = data["runtime"]

    best_length = result["best_length"]

    gap = (
        (best_length - optimal_length)
        / optimal_length
    ) * 100

    summary_rows.append({
        "Configuration": config_name,
        "2-opt Intensity": f"{intensity:.0%}",
        "Best Tour Length": best_length,
        "Optimality Gap (%)": round(gap, 2),
        "Runtime (s)": round(runtime, 2),
        "2-opt Calls": result["two_opt_calls"],
        "Accepted 2-opt Moves": result["two_opt_moves"]
    })

comparison_df = pd.DataFrame(summary_rows)

print("eil51 - Seed 42")
print(comparison_df.to_string(index=False))

In [ ]:
# ==========================================
# STEP 28: CONVERGENCE COMPARISON
# eil51 - Seed 42
# ==========================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for config_name, data in comparison_results.items():

    history = data["result"]["best_history"]

    generations = np.arange(
        1,
        len(history) + 1
    )

    plt.plot(
        generations,
        history,
        label=config_name
    )

# Known optimum
plt.axhline(
    y=OPTIMAL_TOUR_LENGTHS["eil51"],
    linestyle="--",
    label="Known Optimum (426)"
)

plt.xlabel("Generation")
plt.ylabel("Best Tour Length")
plt.title(
    "Convergence Comparison on eil51\n"
    "GA vs. Hybrid GA with Different 2-Opt Intensities"
)

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# STEP 29: CONVERGENCE METRICS
# eil51 - Seed 42
# ==========================================

def first_generation_at_or_below(history, threshold):
    for generation, value in enumerate(history, start=1):
        if value <= threshold:
            return generation
    return None


convergence_rows = []

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

threshold_5pct = optimal_length * 1.05
threshold_1pct = optimal_length * 1.01

for config_name, data in comparison_results.items():

    history = data["result"]["best_history"]
    final_best = data["result"]["best_length"]

    generation_final_best = (
        first_generation_at_or_below(
            history,
            final_best
        )
    )

    generation_5pct = (
        first_generation_at_or_below(
            history,
            threshold_5pct
        )
    )

    generation_1pct = (
        first_generation_at_or_below(
            history,
            threshold_1pct
        )
    )

    convergence_rows.append({
        "Configuration": config_name,
        "Final Best": final_best,
        "Generation Final Best First Reached":
            generation_final_best,
        "Generation Within 5%":
            generation_5pct,
        "Generation Within 1%":
            generation_1pct
    })

convergence_df = pd.DataFrame(convergence_rows)

print("eil51 - Seed 42")
print(convergence_df.to_string(index=False))

In [ ]:
def first_generation_at_or_below(history, threshold):
    for generation, value in enumerate(history, start=1):
        if value <= threshold:
            return generation
    return None

In [ ]:
# ==========================================
# STEP 30A: VALIDATE NESTED 2-OPT SELECTION
# ==========================================

test_seed = 10042

# Simulate one 2-opt activation separately
# for each intensity using the same seed.

def get_selected_indices(
    population_size,
    intensity,
    seed
):
    rng = np.random.default_rng(seed)

    random_order = rng.permutation(
        population_size
    )

    num_selected = int(
        np.floor(
            intensity * population_size
        )
    )

    return random_order[:num_selected]


selected_10 = get_selected_indices(
    100, 0.10, test_seed
)

selected_30 = get_selected_indices(
    100, 0.30, test_seed
)

selected_50 = get_selected_indices(
    100, 0.50, test_seed
)


print("10% count:", len(selected_10))
print("30% count:", len(selected_30))
print("50% count:", len(selected_50))

print(
    "\n10% subset of 30%:",
    set(selected_10).issubset(
        set(selected_30)
    )
)

print(
    "30% subset of 50%:",
    set(selected_30).issubset(
        set(selected_50)
    )
)

print("\nFirst 10 selected indices:")
print(selected_10)

print("\nFirst 10 of 30%:")
print(selected_30[:10])

print("\nFirst 10 of 50%:")
print(selected_50[:10])

In [ ]:
# ==========================================
# STEP 31: FIVE-SEED PILOT
# eil51
# ==========================================

pilot_seeds = [42, 43, 44, 45, 46]

pilot_configurations = {
    "GA": 0.00,
    "GA_2OPT_10": 0.10,
    "GA_2OPT_30": 0.30,
    "GA_2OPT_50": 0.50
}

pilot_rows = []
pilot_histories = {}

optimal_length = OPTIMAL_TOUR_LENGTHS["eil51"]

for seed in pilot_seeds:

    print(f"\n========== Seed {seed} ==========")

    # Separate but reproducible local-search seed
    two_opt_seed = 10000 + seed

    for config_name, intensity in pilot_configurations.items():

        print(
            f"Running {config_name}...",
            end=" "
        )

        start_time = time.perf_counter()

        result = run_hybrid_ga(
            distance_matrix=eil51_distance_matrix,
            two_opt_intensity=intensity,
            population_size=POPULATION_SIZE,
            max_generations=MAX_GENERATIONS,
            crossover_rate=CROSSOVER_RATE,
            mutation_rate=MUTATION_RATE,
            tournament_size=TOURNAMENT_SIZE,
            elite_count=ELITE_COUNT,
            two_opt_interval=TWO_OPT_INTERVAL,
            ga_seed=seed,
            two_opt_seed=two_opt_seed
        )

        runtime = time.perf_counter() - start_time

        best_length = result["best_length"]

        gap = (
            (best_length - optimal_length)
            / optimal_length
        ) * 100

        generation_5pct = first_generation_at_or_below(
            result["best_history"],
            optimal_length * 1.05
        )

        generation_1pct = first_generation_at_or_below(
            result["best_history"],
            optimal_length * 1.01
        )

        generation_final_best = first_generation_at_or_below(
            result["best_history"],
            best_length
        )

        pilot_rows.append({
            "Instance": "eil51",
            "Seed": seed,
            "Configuration": config_name,
            "2-opt Intensity": intensity,
            "Best Tour Length": best_length,
            "Optimality Gap (%)": gap,
            "Runtime (s)": runtime,
            "Generation Within 5%": generation_5pct,
            "Generation Within 1%": generation_1pct,
            "Generation Final Best": generation_final_best,
            "2-opt Calls": result["two_opt_calls"],
            "Accepted 2-opt Moves": result["two_opt_moves"]
        })

        pilot_histories[
            (seed, config_name)
        ] = result["best_history"]

        print(
            f"Best = {best_length}, "
            f"Gap = {gap:.2f}%, "
            f"Time = {runtime:.2f}s"
        )


pilot_df = pd.DataFrame(pilot_rows)

print("\nPilot complete.")
print("Total runs:", len(pilot_df))

In [ ]:
# ==========================================
# STEP 32: INDIVIDUAL PILOT RESULTS
# ==========================================

display_columns = [
    "Seed",
    "Configuration",
    "Best Tour Length",
    "Optimality Gap (%)",
    "Runtime (s)",
    "Generation Within 5%",
    "Generation Within 1%",
    "2-opt Calls",
    "Accepted 2-opt Moves"
]

pilot_display = pilot_df[
    display_columns
].copy()

pilot_display["Optimality Gap (%)"] = (
    pilot_display["Optimality Gap (%)"]
    .round(2)
)

pilot_display["Runtime (s)"] = (
    pilot_display["Runtime (s)"]
    .round(2)
)

print(
    pilot_display.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# STEP 33: PILOT SUMMARY
# ==========================================

pilot_summary = (
    pilot_df
    .groupby("Configuration")
    .agg(
        Mean_Tour_Length=(
            "Best Tour Length",
            "mean"
        ),
        Best_Tour_Length=(
            "Best Tour Length",
            "min"
        ),
        Std_Tour_Length=(
            "Best Tour Length",
            "std"
        ),
        Mean_Gap_Percent=(
            "Optimality Gap (%)",
            "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime (s)",
            "mean"
        ),
        Mean_2Opt_Moves=(
            "Accepted 2-opt Moves",
            "mean"
        )
    )
    .reset_index()
)

pilot_summary[
    "Mean_Tour_Length"
] = pilot_summary[
    "Mean_Tour_Length"
].round(2)

pilot_summary[
    "Std_Tour_Length"
] = pilot_summary[
    "Std_Tour_Length"
].round(2)

pilot_summary[
    "Mean_Gap_Percent"
] = pilot_summary[
    "Mean_Gap_Percent"
].round(2)

pilot_summary[
    "Mean_Runtime_Seconds"
] = pilot_summary[
    "Mean_Runtime_Seconds"
].round(2)

pilot_summary[
    "Mean_2Opt_Moves"
] = pilot_summary[
    "Mean_2Opt_Moves"
].round(1)

print("eil51 Five-Seed Pilot Summary")
print(
    pilot_summary.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# STEP 34: DOWNLOAD REMAINING TSPLIB FILES
# ==========================================

import urllib.request
import os

TSPLIB_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "mastqe/tsplib/master/"
)

remaining_instances = [
    "eil76",
    "kroA100",
    "ch150",
    "kroA200",
    "pr264"
]

for instance_name in remaining_instances:

    filename = f"{instance_name}.tsp"
    url = TSPLIB_BASE_URL + filename

    if not os.path.exists(filename):
        urllib.request.urlretrieve(
            url,
            filename
        )
        print(f"Downloaded: {filename}")
    else:
        print(f"Already exists: {filename}")

In [ ]:
# ==========================================
# STEP 35: INSPECT TSPLIB HEADERS
# ==========================================

all_instances = [
    "eil51",
    "eil76",
    "kroA100",
    "ch150",
    "kroA200",
    "pr264"
]

def read_tsplib_header(filename):

    header = {}

    with open(filename, "r") as file:

        for line in file:

            line = line.strip()

            if line == "NODE_COORD_SECTION":
                break

            if ":" in line:

                key, value = line.split(
                    ":",
                    1
                )

                header[key.strip()] = (
                    value.strip()
                )

    return header


for instance_name in all_instances:

    header = read_tsplib_header(
        f"{instance_name}.tsp"
    )

    print(f"\n{instance_name}")
    print(
        "  TYPE:",
        header.get("TYPE")
    )
    print(
        "  DIMENSION:",
        header.get("DIMENSION")
    )
    print(
        "  EDGE_WEIGHT_TYPE:",
        header.get("EDGE_WEIGHT_TYPE")
    )

In [ ]:
# ==========================================
# STEP 36: LOAD AND VALIDATE ALL INSTANCES
# ==========================================

EXPECTED_DIMENSIONS = {
    "eil51": 51,
    "eil76": 76,
    "kroA100": 100,
    "ch150": 150,
    "kroA200": 200,
    "pr264": 264
}

instance_coordinates = {}
distance_matrices = {}

for instance_name in all_instances:

    filename = f"{instance_name}.tsp"

    coordinates = load_tsplib_coordinates(
        filename
    )

    distance_matrix = build_distance_matrix(
        coordinates
    )

    instance_coordinates[
        instance_name
    ] = coordinates

    distance_matrices[
        instance_name
    ] = distance_matrix

    expected_n = EXPECTED_DIMENSIONS[
        instance_name
    ]

    correct_dimension = (
        len(coordinates) == expected_n
        and distance_matrix.shape
        == (expected_n, expected_n)
    )

    symmetric = np.array_equal(
        distance_matrix,
        distance_matrix.T
    )

    zero_diagonal = np.all(
        np.diag(distance_matrix) == 0
    )

    print(f"\n{instance_name}")
    print(
        "  Cities:",
        len(coordinates)
    )
    print(
        "  Matrix shape:",
        distance_matrix.shape
    )
    print(
        "  Correct dimension:",
        correct_dimension
    )
    print(
        "  Symmetric:",
        symmetric
    )
    print(
        "  Zero diagonal:",
        zero_diagonal
    )
    print(
        "  Known optimum:",
        OPTIMAL_TOUR_LENGTHS[
            instance_name
        ]
    )

In [ ]:
# ==========================================
# STEP 37: MEDIUM-INSTANCE FEASIBILITY PILOT
# kroA100 - Seed 42
# ==========================================

medium_instance = "kroA100"
medium_matrix = distance_matrices[medium_instance]

medium_seed = 42
medium_two_opt_seed = 10000 + medium_seed

medium_results = {}

print(
    f"Instance: {medium_instance}"
)
print(
    f"Cities: {len(medium_matrix)}"
)
print(
    f"Known optimum: "
    f"{OPTIMAL_TOUR_LENGTHS[medium_instance]}"
)

for config_name, intensity in pilot_configurations.items():

    print(
        f"\nRunning {config_name} "
        f"({intensity:.0%})..."
    )

    start_time = time.perf_counter()

    result = run_hybrid_ga(
        distance_matrix=medium_matrix,
        two_opt_intensity=intensity,
        population_size=POPULATION_SIZE,
        max_generations=MAX_GENERATIONS,
        crossover_rate=CROSSOVER_RATE,
        mutation_rate=MUTATION_RATE,
        tournament_size=TOURNAMENT_SIZE,
        elite_count=ELITE_COUNT,
        two_opt_interval=TWO_OPT_INTERVAL,
        ga_seed=medium_seed,
        two_opt_seed=medium_two_opt_seed
    )

    runtime = (
        time.perf_counter() - start_time
    )

    optimal_length = (
        OPTIMAL_TOUR_LENGTHS[
            medium_instance
        ]
    )

    gap = (
        (
            result["best_length"]
            - optimal_length
        )
        / optimal_length
    ) * 100

    medium_results[
        config_name
    ] = {
        "intensity": intensity,
        "result": result,
        "runtime": runtime,
        "gap": gap
    }

    print(
        "Best tour length:",
        result["best_length"]
    )

    print(
        "Optimality gap:",
        round(gap, 2),
        "%"
    )

    print(
        "Runtime:",
        round(runtime, 2),
        "seconds"
    )

    print(
        "2-opt calls:",
        result["two_opt_calls"]
    )

    print(
        "Accepted 2-opt moves:",
        result["two_opt_moves"]
    )

In [ ]:
# ==========================================
# STEP 38: kroA100 FEASIBILITY SUMMARY
# ==========================================

medium_rows = []

for config_name, data in medium_results.items():

    result = data["result"]

    medium_rows.append({
        "Configuration": config_name,
        "2-opt Intensity":
            f"{data['intensity']:.0%}",
        "Best Tour Length":
            result["best_length"],
        "Optimality Gap (%)":
            round(data["gap"], 2),
        "Runtime (s)":
            round(data["runtime"], 2),
        "2-opt Calls":
            result["two_opt_calls"],
        "Accepted 2-opt Moves":
            result["two_opt_moves"]
    })

medium_df = pd.DataFrame(
    medium_rows
)

print(
    "kroA100 - Seed 42 "
    "Feasibility Pilot"
)

print(
    medium_df.to_string(
        index=False
    )
)

In [ ]:
# ==========================================
# STEP 39: LARGE-INSTANCE BASELINE
# pr264 - Seed 42 - GA ONLY
# ==========================================

large_instance = "pr264"
large_matrix = distance_matrices[large_instance]

large_seed = 42
large_two_opt_seed = 10000 + large_seed

print("Instance:", large_instance)
print("Cities:", len(large_matrix))
print(
    "Known optimum:",
    OPTIMAL_TOUR_LENGTHS[large_instance]
)

start_time = time.perf_counter()

pr264_ga_result = run_hybrid_ga(
    distance_matrix=large_matrix,
    two_opt_intensity=0.00,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    two_opt_interval=TWO_OPT_INTERVAL,
    ga_seed=large_seed,
    two_opt_seed=large_two_opt_seed
)

pr264_ga_time = (
    time.perf_counter() - start_time
)

optimal_length = (
    OPTIMAL_TOUR_LENGTHS[large_instance]
)

pr264_ga_gap = (
    (
        pr264_ga_result["best_length"]
        - optimal_length
    )
    / optimal_length
) * 100

print("\nConfiguration: GA (0%)")

print(
    "Best tour length:",
    pr264_ga_result["best_length"]
)

print(
    "Optimality gap:",
    round(pr264_ga_gap, 2),
    "%"
)

print(
    "Runtime:",
    round(pr264_ga_time, 2),
    "seconds"
)

print(
    "2-opt calls:",
    pr264_ga_result["two_opt_calls"]
)

print(
    "Best tour valid:",
    len(pr264_ga_result["best_tour"])
        == len(large_matrix)
    and
    len(set(pr264_ga_result["best_tour"]))
        == len(large_matrix)
)

In [ ]:
# ==========================================
# STEP 40: LARGE-INSTANCE 10% 2-OPT TEST
# pr264 - Seed 42
# ==========================================

print("Instance: pr264")
print("Configuration: GA + 10% 2-opt")
print("Seed: 42")

start_time = time.perf_counter()

pr264_10_result = run_hybrid_ga(
    distance_matrix=large_matrix,
    two_opt_intensity=0.10,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    two_opt_interval=TWO_OPT_INTERVAL,
    ga_seed=large_seed,
    two_opt_seed=large_two_opt_seed
)

pr264_10_time = (
    time.perf_counter() - start_time
)

optimal_length = (
    OPTIMAL_TOUR_LENGTHS["pr264"]
)

pr264_10_gap = (
    (
        pr264_10_result["best_length"]
        - optimal_length
    )
    / optimal_length
) * 100

print(
    "\nBest tour length:",
    pr264_10_result["best_length"]
)

print(
    "Known optimum:",
    optimal_length
)

print(
    "Optimality gap:",
    round(pr264_10_gap, 2),
    "%"
)

print(
    "Runtime:",
    round(pr264_10_time, 2),
    "seconds"
)

print(
    "2-opt calls:",
    pr264_10_result["two_opt_calls"]
)

print(
    "Accepted 2-opt moves:",
    pr264_10_result["two_opt_moves"]
)

print(
    "Best tour valid:",
    len(pr264_10_result["best_tour"]) == 264
    and len(set(pr264_10_result["best_tour"])) == 264
)

In [ ]:
# ==========================================
# SAVE PILOT RESULTS
# ==========================================

pilot_df.to_csv(
    "eil51_five_seed_pilot.csv",
    index=False
)

medium_df.to_csv(
    "kroA100_seed42_pilot.csv",
    index=False
)

print("Pilot results saved.")

In [ ]:
import os
import multiprocessing
import psutil

print("CPU cores available:", os.cpu_count())
print("Logical CPUs:", multiprocessing.cpu_count())

memory_gb = psutil.virtual_memory().total / (1024**3)
print(f"RAM available: {memory_gb:.2f} GB")

In [ ]:
# ==========================================
# STEP 41: VECTORIZED BEST-IMPROVEMENT 2-OPT
# ==========================================

def best_improvement_two_opt_fast(
    tour,
    distance_matrix
):
    current_tour = tour.copy()
    n = len(current_tour)

    current_length = calculate_tour_length(
        current_tour,
        distance_matrix
    )

    num_moves = 0

    while True:

        best_delta = 0
        best_i = None
        best_j = None

        # Examine each first edge (a, b)
        for i in range(n - 2):

            a = current_tour[i]
            b = current_tour[i + 1]

            # Possible second-edge starting positions
            j_indices = np.arange(i + 2, n)

            # Exclude the wrap-around adjacent edge
            if i == 0:
                j_indices = j_indices[
                    j_indices != n - 1
                ]

            if len(j_indices) == 0:
                continue

            c = current_tour[j_indices]

            d = current_tour[
                (j_indices + 1) % n
            ]

            old_distances = (
                distance_matrix[a, b]
                +
                distance_matrix[c, d]
            )

            new_distances = (
                distance_matrix[a, c]
                +
                distance_matrix[b, d]
            )

            deltas = (
                new_distances
                - old_distances
            )

            local_position = np.argmin(
                deltas
            )

            local_delta = int(
                deltas[local_position]
            )

            if local_delta < best_delta:

                best_delta = local_delta

                best_i = i

                best_j = int(
                    j_indices[
                        local_position
                    ]
                )

        # No improving exchange remains
        if best_i is None:
            break

        current_tour[
            best_i + 1 :
            best_j + 1
        ] = current_tour[
            best_i + 1 :
            best_j + 1
        ][::-1]

        current_length += best_delta

        num_moves += 1

    verified_length = calculate_tour_length(
        current_tour,
        distance_matrix
    )

    assert (
        current_length
        == verified_length
    )

    return (
        current_tour,
        int(verified_length),
        num_moves
    )

In [ ]:
# ==========================================
# STEP 42: VALIDATE FAST 2-OPT
# ==========================================

test_rng = np.random.default_rng(12345)

test_tour = test_rng.permutation(
    len(distance_matrices["eil51"])
)

# Original implementation
start = time.perf_counter()

old_tour, old_length, old_moves = (
    best_improvement_two_opt(
        test_tour,
        distance_matrices["eil51"]
    )
)

old_time = time.perf_counter() - start


# Fast implementation
start = time.perf_counter()

fast_tour, fast_length, fast_moves = (
    best_improvement_two_opt_fast(
        test_tour,
        distance_matrices["eil51"]
    )
)

fast_time = time.perf_counter() - start


print("Original 2-opt")
print("Length:", old_length)
print("Moves:", old_moves)
print("Time:", round(old_time, 4), "seconds")

print("\nFast 2-opt")
print("Length:", fast_length)
print("Moves:", fast_moves)
print("Time:", round(fast_time, 4), "seconds")

print(
    "\nSame final length:",
    old_length == fast_length
)

print(
    "Same number of moves:",
    old_moves == fast_moves
)

print(
    "Same final tour:",
    np.array_equal(
        old_tour,
        fast_tour
    )
)

print(
    "Speedup:",
    round(
        old_time / fast_time,
        2
    ),
    "x"
)

In [ ]:
# ==========================================
# STEP 43: ACTIVATE OPTIMIZED 2-OPT
# ==========================================

# Preserve the original implementation
best_improvement_two_opt_original = (
    best_improvement_two_opt
)

# Use the validated optimized implementation
best_improvement_two_opt = (
    best_improvement_two_opt_fast
)

print("Optimized 2-opt activated.")

In [ ]:
# ==========================================
# STEP 44: END-TO-END VALIDATION
# ==========================================

start_time = time.perf_counter()

validation_result = run_hybrid_ga(
    distance_matrix=distance_matrices["eil51"],
    two_opt_intensity=0.30,
    population_size=POPULATION_SIZE,
    max_generations=MAX_GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    tournament_size=TOURNAMENT_SIZE,
    elite_count=ELITE_COUNT,
    two_opt_interval=TWO_OPT_INTERVAL,
    ga_seed=42,
    two_opt_seed=10042
)

validation_time = (
    time.perf_counter() - start_time
)

print(
    "Best length:",
    validation_result["best_length"]
)

print(
    "Expected best length: 427"
)

print(
    "2-opt calls:",
    validation_result["two_opt_calls"]
)

print(
    "Expected calls: 1500"
)

print(
    "Accepted moves:",
    validation_result["two_opt_moves"]
)

print(
    "Expected moves: 2005"
)

print(
    "Runtime:",
    round(validation_time, 2),
    "seconds"
)

print(
    "\nBest length matches:",
    validation_result["best_length"] == 427
)

print(
    "Calls match:",
    validation_result["two_opt_calls"] == 1500
)

print(
    "Moves match:",
    validation_result["two_opt_moves"] == 2005
)

In [ ]:
# ==========================================
# STEP 45: EXPERIMENT OUTPUT DIRECTORIES
# ==========================================

import os
import json
import pandas as pd
import numpy as np
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

RESULTS_DIR = (
    "/content/GA_2Opt_TSP_Experiment/results"
)

RUNS_DIR = os.path.join(
    RESULTS_DIR,
    "individual_runs"
)

HISTORY_DIR = os.path.join(
    RESULTS_DIR,
    "convergence_histories"
)

os.makedirs(
    RUNS_DIR,
    exist_ok=True
)

os.makedirs(
    HISTORY_DIR,
    exist_ok=True
)

print("Results directory:")
print(RESULTS_DIR)

print("\nIndividual runs:")
print(RUNS_DIR)

print("\nConvergence histories:")
print(HISTORY_DIR)

In [ ]:
# ==========================================
# STEP 46: FINAL EXPERIMENT SEEDS
# ==========================================

EXPERIMENT_SEEDS = list(
    range(42, 72)
)

print(
    "Number of experimental seeds:",
    len(EXPERIMENT_SEEDS)
)

print(
    "Seeds:",
    EXPERIMENT_SEEDS
)

In [ ]:
# ==========================================
# STEP 47: SINGLE EXPERIMENTAL RUN
# ==========================================

def run_single_experiment(
    instance_name,
    configuration,
    intensity,
    seed
):

    distance_matrix = (
        distance_matrices[
            instance_name
        ]
    )

    optimum = (
        OPTIMAL_TOUR_LENGTHS[
            instance_name
        ]
    )

    two_opt_seed = (
        10000 + seed
    )

    start_time = (
        time.perf_counter()
    )

    result = run_hybrid_ga(
        distance_matrix=distance_matrix,
        two_opt_intensity=intensity,
        population_size=POPULATION_SIZE,
        max_generations=MAX_GENERATIONS,
        crossover_rate=CROSSOVER_RATE,
        mutation_rate=MUTATION_RATE,
        tournament_size=TOURNAMENT_SIZE,
        elite_count=ELITE_COUNT,
        two_opt_interval=TWO_OPT_INTERVAL,
        ga_seed=seed,
        two_opt_seed=two_opt_seed
    )

    runtime = (
        time.perf_counter()
        - start_time
    )

    best_length = (
        result["best_length"]
    )

    gap = (
        (
            best_length
            - optimum
        )
        / optimum
    ) * 100

    threshold_5 = (
        optimum * 1.05
    )

    threshold_1 = (
        optimum * 1.01
    )

    gen_5 = (
        first_generation_at_or_below(
            result["best_history"],
            threshold_5
        )
    )

    gen_1 = (
        first_generation_at_or_below(
            result["best_history"],
            threshold_1
        )
    )

    # First generation at which
    # final best solution was reached
    gen_final_best = (
        first_generation_at_or_below(
            result["best_history"],
            best_length
        )
    )

    run_record = {
        "Instance": instance_name,
        "Cities": len(distance_matrix),
        "Configuration": configuration,
        "2Opt_Intensity":
            intensity,
        "Seed": seed,
        "Optimal_Tour_Length":
            optimum,
        "Best_Tour_Length":
            best_length,
        "Optimality_Gap_Percent":
            gap,
        "Runtime_Seconds":
            runtime,
        "Generation_Within_5_Percent":
            gen_5,
        "Generation_Within_1_Percent":
            gen_1,
        "Generation_Final_Best":
            gen_final_best,
        "Reached_Optimum":
            best_length == optimum,
        "TwoOpt_Calls":
            result["two_opt_calls"],
        "TwoOpt_Moves":
            result["two_opt_moves"]
    }

    return (
        run_record,
        result["best_history"]
    )

In [ ]:
# ==========================================
# STEP 48: TEST RESULT SAVING
# ==========================================

test_record, test_history = (
    run_single_experiment(
        instance_name="eil51",
        configuration="GA",
        intensity=0.00,
        seed=42
    )
)

print(test_record)
print(
    "History length:",
    len(test_history)
)

In [ ]:
test_file = os.path.join(
    RUNS_DIR,
    "TEST_eil51_GA_seed42.csv"
)

pd.DataFrame(
    [test_record]
).to_csv(
    test_file,
    index=False
)

print(
    "Saved:",
    os.path.exists(test_file)
)

print(test_file)

In [ ]:
os.remove(test_file)
print("Test file removed.")

In [ ]:
# ==========================================
# STEP 49: RESUMABLE EXPERIMENT WORKER
# ==========================================

def experiment_filename(
    instance_name,
    configuration,
    seed
):
    return os.path.join(
        RUNS_DIR,
        f"{instance_name}_{configuration}_seed{seed}.csv"
    )


def history_filename(
    instance_name,
    configuration,
    seed
):
    return os.path.join(
        HISTORY_DIR,
        f"{instance_name}_{configuration}_seed{seed}_history.csv"
    )


def execute_and_save_run(job):

    (
        instance_name,
        configuration,
        intensity,
        seed
    ) = job

    result_file = experiment_filename(
        instance_name,
        configuration,
        seed
    )

    history_file = history_filename(
        instance_name,
        configuration,
        seed
    )

    # Resume protection
    if (
        os.path.exists(result_file)
        and os.path.exists(history_file)
    ):
        return {
            "status": "SKIPPED",
            "instance": instance_name,
            "configuration": configuration,
            "seed": seed
        }

    run_record, history = (
        run_single_experiment(
            instance_name,
            configuration,
            intensity,
            seed
        )
    )

    # Save run result immediately
    pd.DataFrame(
        [run_record]
    ).to_csv(
        result_file,
        index=False
    )

    # Save convergence history
    history_df = pd.DataFrame({
        "Generation": np.arange(
            1,
            len(history) + 1
        ),
        "Best_Tour_Length": history
    })

    history_df.to_csv(
        history_file,
        index=False
    )

    return {
        "status": "COMPLETED",
        "instance": instance_name,
        "configuration": configuration,
        "seed": seed,
        "best_length":
            run_record[
                "Best_Tour_Length"
            ],
        "gap":
            run_record[
                "Optimality_Gap_Percent"
            ],
        "runtime":
            run_record[
                "Runtime_Seconds"
            ]
    }

In [ ]:
# ==========================================
# STEP 50: BUILD FINAL EXPERIMENT
# ==========================================

FINAL_CONFIGURATIONS = {
    "GA": 0.00,
    "GA_2OPT_10": 0.10,
    "GA_2OPT_30": 0.30,
    "GA_2OPT_50": 0.50
}

FINAL_INSTANCES = [
    "eil51",
    "eil76",
    "kroA100",
    "ch150",
    "kroA200",
    "pr264"
]

experiment_jobs = []

for instance_name in FINAL_INSTANCES:
    for seed in EXPERIMENT_SEEDS:
        for (
            configuration,
            intensity
        ) in FINAL_CONFIGURATIONS.items():

            experiment_jobs.append(
                (
                    instance_name,
                    configuration,
                    intensity,
                    seed
                )
            )

print(
    "Total experimental runs:",
    len(experiment_jobs)
)

assert len(experiment_jobs) == 720

In [ ]:
# ==========================================
# STEP 51: RUN FINAL EXPERIMENT
# 2 CPU WORKERS + AUTOMATIC CHECKPOINTING
# ==========================================

MAX_WORKERS = 2

completed = 0
skipped = 0
failed = 0

experiment_start = time.perf_counter()

print("FINAL EXPERIMENT STARTED")
print("========================")
print("Total runs:", len(experiment_jobs))
print("CPU workers:", MAX_WORKERS)
print()


with ProcessPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    future_to_job = {
        executor.submit(
            execute_and_save_run,
            job
        ): job
        for job in experiment_jobs
    }

    for future in as_completed(
        future_to_job
    ):

        job = future_to_job[future]

        try:
            output = future.result()

            if output["status"] == "SKIPPED":

                skipped += 1

                print(
                    f"[SKIPPED] "
                    f"{output['instance']} | "
                    f"{output['configuration']} | "
                    f"seed {output['seed']}"
                )

            else:

                completed += 1

                print(
                    f"[{completed + skipped}/720] "
                    f"{output['instance']} | "
                    f"{output['configuration']} | "
                    f"seed {output['seed']} | "
                    f"best={output['best_length']} | "
                    f"gap={output['gap']:.2f}% | "
                    f"time={output['runtime']:.1f}s"
                )

        except Exception as error:

            failed += 1

            print(
                "[FAILED]",
                job,
                "|",
                repr(error)
            )


total_time = (
    time.perf_counter()
    - experiment_start
)

print("\n========================")
print("EXPERIMENT SESSION ENDED")
print("========================")

print(
    "Completed this session:",
    completed
)

print(
    "Previously completed:",
    skipped
)

print(
    "Failed:",
    failed
)

print(
    "Elapsed hours:",
    round(
        total_time / 3600,
        2
    )
)

In [ ]:
# ==========================================
# STEP 52: VERIFY COMPLETE EXPERIMENT FILES
# ==========================================

import os

run_files = [
    f for f in os.listdir(RUNS_DIR)
    if f.endswith(".csv")
    and not f.startswith("TEST_")
]

history_files = [
    f for f in os.listdir(HISTORY_DIR)
    if f.endswith(".csv")
]

print("Individual run files:", len(run_files))
print("Convergence history files:", len(history_files))

print(
    "\nExpected individual runs: 720"
)

print(
    "Expected convergence histories: 720"
)

assert len(run_files) == 720, (
    f"Expected 720 run files, found {len(run_files)}"
)

assert len(history_files) == 720, (
    f"Expected 720 history files, found {len(history_files)}"
)

print("\n✅ COMPLETE EXPERIMENT VERIFIED")
print("All 720 runs and 720 histories are present.")

In [ ]:
# ==========================================
# STEP 53: BUILD MASTER RESULTS DATASET
# ==========================================

import os
import pandas as pd

run_files = sorted([
    f for f in os.listdir(RUNS_DIR)
    if f.endswith(".csv")
    and not f.startswith("TEST_")
])

master_frames = []

for filename in run_files:

    filepath = os.path.join(
        RUNS_DIR,
        filename
    )

    df = pd.read_csv(filepath)

    master_frames.append(df)

master_results = pd.concat(
    master_frames,
    ignore_index=True
)

# Sort in experimental order
instance_order = [
    "eil51",
    "eil76",
    "kroA100",
    "ch150",
    "kroA200",
    "pr264"
]

configuration_order = [
    "GA",
    "GA_2OPT_10",
    "GA_2OPT_30",
    "GA_2OPT_50"
]

master_results["Instance"] = pd.Categorical(
    master_results["Instance"],
    categories=instance_order,
    ordered=True
)

master_results["Configuration"] = pd.Categorical(
    master_results["Configuration"],
    categories=configuration_order,
    ordered=True
)

master_results = (
    master_results
    .sort_values(
        ["Instance", "Seed", "Configuration"]
    )
    .reset_index(drop=True)
)

# Save master dataset
MASTER_RESULTS_FILE = os.path.join(
    RESULTS_DIR,
    "master_experiment_results.csv"
)

master_results.to_csv(
    MASTER_RESULTS_FILE,
    index=False
)

print("Master dataset created.")
print("Rows:", len(master_results))
print("Columns:", len(master_results.columns))
print("Saved to:")
print(MASTER_RESULTS_FILE)

print("\nRuns by instance:")
print(
    master_results[
        "Instance"
    ].value_counts(sort=False)
)

print("\nRuns by configuration:")
print(
    master_results[
        "Configuration"
    ].value_counts(sort=False)
)

print("\nFirst 10 rows:")
print(
    master_results.head(10).to_string(
        index=False
    )
)

assert len(master_results) == 720

print("\n✅ Master dataset contains all 720 experimental runs.")

In [ ]:
# ==========================================
# STEP 54: VALIDATE MASTER DATASET
# ==========================================

print("MASTER DATASET VALIDATION")
print("=========================\n")

# ------------------------------------------
# 1. Check total rows
# ------------------------------------------

print("Total rows:", len(master_results))

assert len(master_results) == 720


# ------------------------------------------
# 2. Check instance × configuration counts
# ------------------------------------------

cell_counts = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    .size()
    .unstack()
)

print("\nRuns per Instance × Configuration:")
print(cell_counts)

assert (cell_counts == 30).all().all()


# ------------------------------------------
# 3. Check duplicate experimental runs
# ------------------------------------------

duplicate_mask = master_results.duplicated(
    subset=[
        "Instance",
        "Configuration",
        "Seed"
    ],
    keep=False
)

duplicate_count = duplicate_mask.sum()

print(
    "\nDuplicate Instance × Configuration × Seed rows:",
    duplicate_count
)

assert duplicate_count == 0


# ------------------------------------------
# 4. Check seeds
# ------------------------------------------

seed_check = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )["Seed"]
    .nunique()
)

print(
    "\nMinimum unique seeds per condition:",
    seed_check.min()
)

print(
    "Maximum unique seeds per condition:",
    seed_check.max()
)

assert seed_check.min() == 30
assert seed_check.max() == 30


# ------------------------------------------
# 5. Check core fields for missing values
# ------------------------------------------

core_columns = [
    "Instance",
    "Cities",
    "Configuration",
    "2Opt_Intensity",
    "Seed",
    "Optimal_Tour_Length",
    "Best_Tour_Length",
    "Optimality_Gap_Percent",
    "Runtime_Seconds",
    "Reached_Optimum",
    "TwoOpt_Calls",
    "TwoOpt_Moves"
]

core_missing = (
    master_results[
        core_columns
    ]
    .isna()
    .sum()
)

print("\nMissing values in core fields:")
print(core_missing)

assert core_missing.sum() == 0


# ------------------------------------------
# 6. Convergence NaNs
# These are allowed when threshold not reached
# ------------------------------------------

convergence_columns = [
    "Generation_Within_5_Percent",
    "Generation_Within_1_Percent",
    "Generation_Final_Best"
]

print("\nMissing convergence values:")
print(
    master_results[
        convergence_columns
    ]
    .isna()
    .sum()
)


# ------------------------------------------
# 7. Basic validity checks
# ------------------------------------------

assert (
    master_results["Best_Tour_Length"]
    >= master_results["Optimal_Tour_Length"]
).all()

assert (
    master_results["Optimality_Gap_Percent"]
    >= 0
).all()

assert (
    master_results["Runtime_Seconds"]
    > 0
).all()

assert (
    master_results["Generation_Final_Best"]
    .between(1, MAX_GENERATIONS)
    .all()
)

print("\nBasic numerical validity checks: PASSED")


# ------------------------------------------
# 8. Check expected 2-opt intensity mapping
# ------------------------------------------

expected_intensities = {
    "GA": 0.00,
    "GA_2OPT_10": 0.10,
    "GA_2OPT_30": 0.30,
    "GA_2OPT_50": 0.50
}

intensity_errors = 0

for config, expected in expected_intensities.items():

    values = (
        master_results.loc[
            master_results[
                "Configuration"
            ] == config,
            "2Opt_Intensity"
        ]
        .unique()
    )

    if not (
        len(values) == 1
        and np.isclose(
            values[0],
            expected
        )
    ):
        intensity_errors += 1

print(
    "\nConfiguration/intensity mapping errors:",
    intensity_errors
)

assert intensity_errors == 0


# ------------------------------------------
# 9. Check GA has no 2-opt activity
# ------------------------------------------

ga_rows = master_results[
    master_results["Configuration"] == "GA"
]

assert (ga_rows["TwoOpt_Calls"] == 0).all()
assert (ga_rows["TwoOpt_Moves"] == 0).all()

print("GA 2-opt activity check: PASSED")


print("\n=========================")
print("✅ MASTER DATASET VALIDATED")
print("=========================")

In [ ]:
# ==========================================
# STEP 55: DESCRIPTIVE RESULTS SUMMARY
# ==========================================

summary_results = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    .agg(
        Runs=(
            "Seed",
            "count"
        ),

        Mean_Best_Tour=(
            "Best_Tour_Length",
            "mean"
        ),

        SD_Best_Tour=(
            "Best_Tour_Length",
            "std"
        ),

        Best_Tour=(
            "Best_Tour_Length",
            "min"
        ),

        Worst_Tour=(
            "Best_Tour_Length",
            "max"
        ),

        Mean_Gap_Percent=(
            "Optimality_Gap_Percent",
            "mean"
        ),

        SD_Gap_Percent=(
            "Optimality_Gap_Percent",
            "std"
        ),

        Mean_Runtime_Seconds=(
            "Runtime_Seconds",
            "mean"
        ),

        SD_Runtime_Seconds=(
            "Runtime_Seconds",
            "std"
        ),

        Mean_Final_Best_Generation=(
            "Generation_Final_Best",
            "mean"
        ),

        Optimum_Reached_Count=(
            "Reached_Optimum",
            "sum"
        ),

        Mean_TwoOpt_Moves=(
            "TwoOpt_Moves",
            "mean"
        )
    )
    .reset_index()
)

# Percentage of runs reaching the known optimum
summary_results[
    "Optimum_Reached_Percent"
] = (
    summary_results[
        "Optimum_Reached_Count"
    ]
    /
    summary_results["Runs"]
    * 100
)

# Round display values
display_summary = (
    summary_results.copy()
)

columns_to_round = [
    "Mean_Best_Tour",
    "SD_Best_Tour",
    "Mean_Gap_Percent",
    "SD_Gap_Percent",
    "Mean_Runtime_Seconds",
    "SD_Runtime_Seconds",
    "Mean_Final_Best_Generation",
    "Mean_TwoOpt_Moves",
    "Optimum_Reached_Percent"
]

display_summary[
    columns_to_round
] = (
    display_summary[
        columns_to_round
    ].round(2)
)

print(
    display_summary.to_string(
        index=False
    )
)

# Save summary
SUMMARY_FILE = os.path.join(
    RESULTS_DIR,
    "descriptive_summary.csv"
)

summary_results.to_csv(
    SUMMARY_FILE,
    index=False
)

print("\nSaved to:")
print(SUMMARY_FILE)

assert len(summary_results) == 24

print(
    "\n✅ 24 Instance × Configuration "
    "summary rows created."
)

In [ ]:
# ==========================================
# STEP 56: CONVERGENCE THRESHOLD SUMMARY
# ==========================================

convergence_summary_rows = []

for (
    instance_name,
    configuration
), group in master_results.groupby(
    ["Instance", "Configuration"],
    observed=True
):

    total_runs = len(group)

    # -------------------------------
    # Within 5%
    # -------------------------------

    reached_5 = (
        group[
            "Generation_Within_5_Percent"
        ]
        .notna()
    )

    count_5 = reached_5.sum()

    percent_5 = (
        count_5 / total_runs
    ) * 100

    mean_gen_5 = (
        group.loc[
            reached_5,
            "Generation_Within_5_Percent"
        ]
        .mean()
    )

    # -------------------------------
    # Within 1%
    # -------------------------------

    reached_1 = (
        group[
            "Generation_Within_1_Percent"
        ]
        .notna()
    )

    count_1 = reached_1.sum()

    percent_1 = (
        count_1 / total_runs
    ) * 100

    mean_gen_1 = (
        group.loc[
            reached_1,
            "Generation_Within_1_Percent"
        ]
        .mean()
    )

    convergence_summary_rows.append({
        "Instance": instance_name,
        "Configuration": configuration,

        "Runs": total_runs,

        "Reached_5pct_Count":
            count_5,

        "Reached_5pct_Percent":
            percent_5,

        "Mean_Generation_5pct_Successful":
            mean_gen_5,

        "Reached_1pct_Count":
            count_1,

        "Reached_1pct_Percent":
            percent_1,

        "Mean_Generation_1pct_Successful":
            mean_gen_1
    })


convergence_summary = pd.DataFrame(
    convergence_summary_rows
)

# Preserve experimental ordering
convergence_summary["Instance"] = pd.Categorical(
    convergence_summary["Instance"],
    categories=FINAL_INSTANCES,
    ordered=True
)

convergence_summary["Configuration"] = pd.Categorical(
    convergence_summary["Configuration"],
    categories=list(
        FINAL_CONFIGURATIONS.keys()
    ),
    ordered=True
)

convergence_summary = (
    convergence_summary
    .sort_values(
        ["Instance", "Configuration"]
    )
    .reset_index(drop=True)
)


# Display copy
display_convergence = (
    convergence_summary.copy()
)

round_columns = [
    "Reached_5pct_Percent",
    "Mean_Generation_5pct_Successful",
    "Reached_1pct_Percent",
    "Mean_Generation_1pct_Successful"
]

display_convergence[
    round_columns
] = (
    display_convergence[
        round_columns
    ].round(2)
)

print(
    display_convergence.to_string(
        index=False
    )
)


# Save
CONVERGENCE_SUMMARY_FILE = os.path.join(
    RESULTS_DIR,
    "convergence_threshold_summary.csv"
)

convergence_summary.to_csv(
    CONVERGENCE_SUMMARY_FILE,
    index=False
)

print("\nSaved to:")
print(CONVERGENCE_SUMMARY_FILE)

assert len(convergence_summary) == 24

print(
    "\n✅ Convergence threshold summary complete."
)

In [ ]:
# ==========================================
# STEP 57: MATCHED QUALITY IMPROVEMENT VS GA
# ==========================================

quality_wide = (
    master_results
    .pivot(
        index=["Instance", "Seed"],
        columns="Configuration",
        values="Best_Tour_Length"
    )
    .reset_index()
)

hybrid_configs = [
    "GA_2OPT_10",
    "GA_2OPT_30",
    "GA_2OPT_50"
]

improvement_rows = []

for config in hybrid_configs:

    improvement = (
        (
            quality_wide["GA"]
            - quality_wide[config]
        )
        / quality_wide["GA"]
        * 100
    )

    temp = pd.DataFrame({
        "Instance":
            quality_wide["Instance"],
        "Seed":
            quality_wide["Seed"],
        "Configuration":
            config,
        "Improvement_vs_GA_Percent":
            improvement
    })

    improvement_rows.append(temp)

matched_improvement = pd.concat(
    improvement_rows,
    ignore_index=True
)

improvement_summary = (
    matched_improvement
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    .agg(
        Mean_Improvement_Percent=(
            "Improvement_vs_GA_Percent",
            "mean"
        ),
        SD_Improvement_Percent=(
            "Improvement_vs_GA_Percent",
            "std"
        ),
        Min_Improvement_Percent=(
            "Improvement_vs_GA_Percent",
            "min"
        ),
        Max_Improvement_Percent=(
            "Improvement_vs_GA_Percent",
            "max"
        )
    )
    .reset_index()
)

display_improvement = (
    improvement_summary.copy()
)

numeric_columns = [
    "Mean_Improvement_Percent",
    "SD_Improvement_Percent",
    "Min_Improvement_Percent",
    "Max_Improvement_Percent"
]

display_improvement[
    numeric_columns
] = (
    display_improvement[
        numeric_columns
    ].round(2)
)

print(
    display_improvement.to_string(
        index=False
    )
)

# Save
IMPROVEMENT_FILE = os.path.join(
    RESULTS_DIR,
    "matched_quality_improvement_vs_GA.csv"
)

matched_improvement.to_csv(
    IMPROVEMENT_FILE,
    index=False
)

print("\nSaved to:")
print(IMPROVEMENT_FILE)

print(
    "\n✅ Matched GA improvement analysis complete."
)

In [ ]:
# ==========================================
# STEP 58: FRIEDMAN TEST
# SOLUTION QUALITY / OPTIMALITY GAP
# ==========================================

from scipy.stats import friedmanchisquare

friedman_quality_rows = []

for instance_name in FINAL_INSTANCES:

    instance_data = (
        master_results[
            master_results["Instance"]
            == instance_name
        ]
    )

    gap_wide = (
        instance_data
        .pivot(
            index="Seed",
            columns="Configuration",
            values="Optimality_Gap_Percent"
        )
        .sort_index()
    )

    statistic, p_value = friedmanchisquare(
        gap_wide["GA"],
        gap_wide["GA_2OPT_10"],
        gap_wide["GA_2OPT_30"],
        gap_wide["GA_2OPT_50"]
    )

    # Kendall's W effect size
    n = len(gap_wide)
    k = 4

    kendalls_w = (
        statistic
        / (n * (k - 1))
    )

    friedman_quality_rows.append({
        "Instance": instance_name,
        "N": n,
        "Friedman_ChiSquare": statistic,
        "df": k - 1,
        "p_value": p_value,
        "Kendalls_W": kendalls_w
    })


friedman_quality = pd.DataFrame(
    friedman_quality_rows
)

display_friedman = (
    friedman_quality.copy()
)

display_friedman[
    "Friedman_ChiSquare"
] = (
    display_friedman[
        "Friedman_ChiSquare"
    ].round(3)
)

display_friedman[
    "Kendalls_W"
] = (
    display_friedman[
        "Kendalls_W"
    ].round(3)
)

print(
    display_friedman.to_string(
        index=False
    )
)


# Save results
FRIEDMAN_QUALITY_FILE = os.path.join(
    RESULTS_DIR,
    "friedman_solution_quality.csv"
)

friedman_quality.to_csv(
    FRIEDMAN_QUALITY_FILE,
    index=False
)

print("\nSaved to:")
print(FRIEDMAN_QUALITY_FILE)

print(
    "\n✅ Friedman solution-quality tests complete."
)

In [ ]:
# ==========================================
# STEP 59: WILCOXON POST-HOC TESTS
# SOLUTION QUALITY
# HOLM CORRECTION
# ==========================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

quality_pairs = list(
    combinations(
        [
            "GA",
            "GA_2OPT_10",
            "GA_2OPT_30",
            "GA_2OPT_50"
        ],
        2
    )
)

wilcoxon_quality_rows = []

for instance_name in FINAL_INSTANCES:

    instance_data = (
        master_results[
            master_results["Instance"]
            == instance_name
        ]
    )

    gap_wide = (
        instance_data
        .pivot(
            index="Seed",
            columns="Configuration",
            values="Optimality_Gap_Percent"
        )
        .sort_index()
    )

    instance_results = []

    for config_a, config_b in quality_pairs:

        x = gap_wide[config_a]
        y = gap_wide[config_b]

        statistic, p_value = wilcoxon(
            x,
            y,
            alternative="two-sided"
        )

        # Direction based on mean gap.
        # Lower gap = better solution quality.
        mean_a = x.mean()
        mean_b = y.mean()

        if mean_a < mean_b:
            better = config_a
        elif mean_b < mean_a:
            better = config_b
        else:
            better = "Tie"

        instance_results.append({
            "Instance": instance_name,
            "Comparison":
                f"{config_a} vs {config_b}",
            "Config_A": config_a,
            "Config_B": config_b,
            "Mean_Gap_A": mean_a,
            "Mean_Gap_B": mean_b,
            "Better_Mean": better,
            "Wilcoxon_Statistic": statistic,
            "Raw_p": p_value
        })

    # Holm correction across the six
    # comparisons for this instance
    raw_p_values = [
        row["Raw_p"]
        for row in instance_results
    ]

    reject, corrected_p, _, _ = (
        multipletests(
            raw_p_values,
            alpha=0.05,
            method="holm"
        )
    )

    for row, reject_null, adjusted_p in zip(
        instance_results,
        reject,
        corrected_p
    ):

        row["Holm_Adjusted_p"] = (
            adjusted_p
        )

        row["Significant_0.05"] = (
            bool(reject_null)
        )

        wilcoxon_quality_rows.append(
            row
        )


wilcoxon_quality = pd.DataFrame(
    wilcoxon_quality_rows
)

display_wilcoxon = (
    wilcoxon_quality.copy()
)

display_wilcoxon[
    [
        "Mean_Gap_A",
        "Mean_Gap_B"
    ]
] = (
    display_wilcoxon[
        [
            "Mean_Gap_A",
            "Mean_Gap_B"
        ]
    ].round(3)
)

print(
    display_wilcoxon[
        [
            "Instance",
            "Comparison",
            "Mean_Gap_A",
            "Mean_Gap_B",
            "Better_Mean",
            "Wilcoxon_Statistic",
            "Raw_p",
            "Holm_Adjusted_p",
            "Significant_0.05"
        ]
    ].to_string(index=False)
)


# Save full results
WILCOXON_QUALITY_FILE = os.path.join(
    RESULTS_DIR,
    "wilcoxon_solution_quality_holm.csv"
)

wilcoxon_quality.to_csv(
    WILCOXON_QUALITY_FILE,
    index=False
)

print("\nSaved to:")
print(WILCOXON_QUALITY_FILE)

print(
    "\n✅ Wilcoxon post-hoc solution-quality "
    "tests complete."
)

In [ ]:
# ==========================================
# STEP 60: RUNTIME STATISTICAL ANALYSIS
# FRIEDMAN + WILCOXON + HOLM
# ==========================================

from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# ------------------------------------------
# PART A: FRIEDMAN TEST
# ------------------------------------------

friedman_runtime_rows = []

for instance_name in FINAL_INSTANCES:

    instance_data = master_results[
        master_results["Instance"] == instance_name
    ]

    runtime_wide = (
        instance_data
        .pivot(
            index="Seed",
            columns="Configuration",
            values="Runtime_Seconds"
        )
        .sort_index()
    )

    statistic, p_value = friedmanchisquare(
        runtime_wide["GA"],
        runtime_wide["GA_2OPT_10"],
        runtime_wide["GA_2OPT_30"],
        runtime_wide["GA_2OPT_50"]
    )

    n = len(runtime_wide)
    k = 4

    kendalls_w = (
        statistic
        / (n * (k - 1))
    )

    friedman_runtime_rows.append({
        "Instance": instance_name,
        "N": n,
        "Friedman_ChiSquare": statistic,
        "df": k - 1,
        "p_value": p_value,
        "Kendalls_W": kendalls_w
    })


friedman_runtime = pd.DataFrame(
    friedman_runtime_rows
)

print("FRIEDMAN TEST — RUNTIME")
print("=======================\n")

print(
    friedman_runtime.to_string(
        index=False
    )
)


# ------------------------------------------
# PART B: WILCOXON POST-HOC
# ------------------------------------------

runtime_pairs = list(
    combinations(
        [
            "GA",
            "GA_2OPT_10",
            "GA_2OPT_30",
            "GA_2OPT_50"
        ],
        2
    )
)

wilcoxon_runtime_rows = []

for instance_name in FINAL_INSTANCES:

    instance_data = master_results[
        master_results["Instance"] == instance_name
    ]

    runtime_wide = (
        instance_data
        .pivot(
            index="Seed",
            columns="Configuration",
            values="Runtime_Seconds"
        )
        .sort_index()
    )

    instance_results = []

    for config_a, config_b in runtime_pairs:

        x = runtime_wide[config_a]
        y = runtime_wide[config_b]

        statistic, p_value = wilcoxon(
            x,
            y,
            alternative="two-sided"
        )

        mean_a = x.mean()
        mean_b = y.mean()

        instance_results.append({
            "Instance": instance_name,
            "Comparison":
                f"{config_a} vs {config_b}",
            "Mean_Runtime_A":
                mean_a,
            "Mean_Runtime_B":
                mean_b,
            "Runtime_Difference":
                mean_b - mean_a,
            "Wilcoxon_Statistic":
                statistic,
            "Raw_p":
                p_value
        })

    raw_p_values = [
        row["Raw_p"]
        for row in instance_results
    ]

    reject, corrected_p, _, _ = (
        multipletests(
            raw_p_values,
            alpha=0.05,
            method="holm"
        )
    )

    for row, reject_null, adjusted_p in zip(
        instance_results,
        reject,
        corrected_p
    ):

        row["Holm_Adjusted_p"] = adjusted_p

        row["Significant_0.05"] = bool(
            reject_null
        )

        wilcoxon_runtime_rows.append(row)


wilcoxon_runtime = pd.DataFrame(
    wilcoxon_runtime_rows
)


print("\n\nWILCOXON POST-HOC — RUNTIME")
print("===========================\n")

display_runtime = (
    wilcoxon_runtime.copy()
)

display_runtime[
    [
        "Mean_Runtime_A",
        "Mean_Runtime_B",
        "Runtime_Difference"
    ]
] = (
    display_runtime[
        [
            "Mean_Runtime_A",
            "Mean_Runtime_B",
            "Runtime_Difference"
        ]
    ].round(2)
)

print(
    display_runtime.to_string(
        index=False
    )
)


# ------------------------------------------
# SAVE
# ------------------------------------------

FRIEDMAN_RUNTIME_FILE = os.path.join(
    RESULTS_DIR,
    "friedman_runtime.csv"
)

WILCOXON_RUNTIME_FILE = os.path.join(
    RESULTS_DIR,
    "wilcoxon_runtime_holm.csv"
)

friedman_runtime.to_csv(
    FRIEDMAN_RUNTIME_FILE,
    index=False
)

wilcoxon_runtime.to_csv(
    WILCOXON_RUNTIME_FILE,
    index=False
)

print("\nSaved:")
print(FRIEDMAN_RUNTIME_FILE)
print(WILCOXON_RUNTIME_FILE)

print(
    "\n✅ Runtime statistical analysis complete."
)

In [ ]:
# ==========================================
# STEP 61: RUNTIME COST INCREASE
# ==========================================

runtime_wide = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )["Runtime_Seconds"]
    .mean()
    .unstack()
)

runtime_cost_rows = []

for instance_name in FINAL_INSTANCES:

    ga = runtime_wide.loc[
        instance_name, "GA"
    ]

    r10 = runtime_wide.loc[
        instance_name, "GA_2OPT_10"
    ]

    r30 = runtime_wide.loc[
        instance_name, "GA_2OPT_30"
    ]

    r50 = runtime_wide.loc[
        instance_name, "GA_2OPT_50"
    ]

    runtime_cost_rows.append({
        "Instance": instance_name,

        "GA_Runtime": ga,
        "10pct_Runtime": r10,
        "30pct_Runtime": r30,
        "50pct_Runtime": r50,

        "10pct_vs_GA_Percent":
            ((r10 - ga) / ga) * 100,

        "30pct_vs_GA_Percent":
            ((r30 - ga) / ga) * 100,

        "50pct_vs_GA_Percent":
            ((r50 - ga) / ga) * 100,

        "30pct_vs_10pct_Percent":
            ((r30 - r10) / r10) * 100,

        "50pct_vs_30pct_Percent":
            ((r50 - r30) / r30) * 100
    })


runtime_cost_summary = pd.DataFrame(
    runtime_cost_rows
)

display_runtime_cost = (
    runtime_cost_summary.copy()
)

numeric_cols = [
    col
    for col in display_runtime_cost.columns
    if col != "Instance"
]

display_runtime_cost[
    numeric_cols
] = (
    display_runtime_cost[
        numeric_cols
    ].round(2)
)

print(
    display_runtime_cost.to_string(
        index=False
    )
)


# Save
RUNTIME_COST_FILE = os.path.join(
    RESULTS_DIR,
    "runtime_cost_summary.csv"
)

runtime_cost_summary.to_csv(
    RUNTIME_COST_FILE,
    index=False
)

print("\nSaved to:")
print(RUNTIME_COST_FILE)

print(
    "\n✅ Runtime cost summary complete."
)

In [ ]:
# ==========================================
# STEP 62: FIGURE 1
# MEAN OPTIMALITY GAP BY CONFIGURATION
# ==========================================

import matplotlib.pyplot as plt

config_order = [
    "GA",
    "GA_2OPT_10",
    "GA_2OPT_30",
    "GA_2OPT_50"
]

config_labels = [
    "GA",
    "10%",
    "30%",
    "50%"
]

figure1_data = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    ["Optimality_Gap_Percent"]
    .mean()
    .unstack()
)

figure1_data = figure1_data.loc[
    FINAL_INSTANCES,
    config_order
]

plt.figure(figsize=(10, 6))

for instance_name in FINAL_INSTANCES:

    plt.plot(
        config_labels,
        figure1_data.loc[
            instance_name,
            config_order
        ],
        marker="o",
        linewidth=2,
        label=instance_name
    )

plt.yscale("log")

plt.xlabel("2-Opt Population Coverage")
plt.ylabel("Mean Optimality Gap (%) — Log Scale")

plt.title(
    "Effect of 2-Opt Intensity on Solution Quality"
)

plt.grid(
    True,
    which="both",
    alpha=0.3
)

plt.legend(
    title="TSPLIB Instance",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

FIGURE1_FILE = os.path.join(
    RESULTS_DIR,
    "figure1_mean_optimality_gap.png"
)

plt.savefig(
    FIGURE1_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(FIGURE1_FILE)

In [ ]:
# ==========================================
# STEP 63: FIGURE 2
# MEAN RUNTIME BY CONFIGURATION
# ==========================================

figure2_data = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    ["Runtime_Seconds"]
    .mean()
    .unstack()
)

figure2_data = figure2_data.loc[
    FINAL_INSTANCES,
    config_order
]

plt.figure(figsize=(10, 6))

for instance_name in FINAL_INSTANCES:

    plt.plot(
        config_labels,
        figure2_data.loc[
            instance_name,
            config_order
        ],
        marker="o",
        linewidth=2,
        label=instance_name
    )

plt.xlabel("2-Opt Population Coverage")
plt.ylabel("Mean Runtime (Seconds)")

plt.title(
    "Effect of 2-Opt Intensity on Computational Runtime"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend(
    title="TSPLIB Instance",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

FIGURE2_FILE = os.path.join(
    RESULTS_DIR,
    "figure2_mean_runtime.png"
)

plt.savefig(
    FIGURE2_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(FIGURE2_FILE)

In [ ]:
# ==========================================
# STEP 64: LOAD CONVERGENCE HISTORIES
# ==========================================

import os
import glob
import pandas as pd

# Recreate the convergence-history path
HISTORIES_DIR = (
    "/content/GA_2Opt_TSP_Experiment/results/convergence_histories"
)

print("History directory:")
print(HISTORIES_DIR)

# Find all convergence-history CSV files
history_files = glob.glob(
    os.path.join(
        HISTORIES_DIR,
        "*.csv"
    )
)

print(
    "\nConvergence history files found:",
    len(history_files)
)

assert len(history_files) == 720, (
    f"Expected 720 history files, "
    f"but found {len(history_files)}."
)

# Load all files
all_histories = []

for file_path in history_files:

    history_df = pd.read_csv(
        file_path
    )

    all_histories.append(
        history_df
    )

# Combine them
convergence_data = pd.concat(
    all_histories,
    ignore_index=True
)

print("\nColumns:")
print(
    convergence_data.columns.tolist()
)

print("\nTotal rows:")
print(
    len(convergence_data)
)

print("\nFirst five rows:")
print(
    convergence_data.head()
)

print(
    "\n✅ Convergence histories loaded successfully."
)

In [ ]:
# ==========================================
# STEP 65A: INSPECT HISTORY FILENAMES
# ==========================================

print("First 12 history filenames:\n")

for file_path in sorted(history_files)[:12]:
    print(
        os.path.basename(file_path)
    )

In [ ]:
# ==========================================
# STEP 65B: BUILD COMPLETE
# CONVERGENCE DATASET
# ==========================================

import re

convergence_frames = []

known_configs = [
    "GA_2OPT_10",
    "GA_2OPT_30",
    "GA_2OPT_50",
    "GA"
]

for file_path in history_files:

    filename = os.path.basename(
        file_path
    )

    # -----------------------------
    # Identify configuration
    # -----------------------------
    configuration = None

    for config in known_configs:
        if f"_{config}_" in filename:
            configuration = config
            break

    if configuration is None:
        raise ValueError(
            f"Could not identify configuration: "
            f"{filename}"
        )

    # -----------------------------
    # Identify instance
    # -----------------------------
    instance_name = filename.split(
        f"_{configuration}_"
    )[0]

    # -----------------------------
    # Identify seed
    # -----------------------------
    seed_match = re.search(
        r"_seed(\d+)_history\.csv$",
        filename
    )

    if seed_match is None:
        raise ValueError(
            f"Could not identify seed: "
            f"{filename}"
        )

    seed = int(
        seed_match.group(1)
    )

    # -----------------------------
    # Load history
    # -----------------------------
    df = pd.read_csv(
        file_path
    )

    df["Instance"] = instance_name
    df["Configuration"] = configuration
    df["Seed"] = seed

    # Known optimum
    optimum = OPTIMAL_TOUR_LENGTHS[
        instance_name
    ]

    df["Optimal_Tour_Length"] = optimum

    # Convert tour length to gap %
    df["Optimality_Gap_Percent"] = (
        (
            df["Best_Tour_Length"]
            - optimum
        )
        / optimum
        * 100
    )

    convergence_frames.append(df)


# Combine all 720 runs
convergence_full = pd.concat(
    convergence_frames,
    ignore_index=True
)


# ==========================================
# VALIDATION
# ==========================================

print(
    "Total convergence rows:",
    len(convergence_full)
)

print(
    "\nRuns represented:",
    convergence_full[
        [
            "Instance",
            "Configuration",
            "Seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nRows per Instance × Configuration:"
)

print(
    convergence_full
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    .size()
    .unstack()
)

print(
    "\nGeneration range:",
    convergence_full["Generation"].min(),
    "to",
    convergence_full["Generation"].max()
)

print(
    "\nGap range:",
    round(
        convergence_full[
            "Optimality_Gap_Percent"
        ].min(),
        4
    ),
    "to",
    round(
        convergence_full[
            "Optimality_Gap_Percent"
        ].max(),
        4
    )
)


# Important checks
assert len(convergence_full) == 360000

assert (
    convergence_full[
        [
            "Instance",
            "Configuration",
            "Seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 720
)

assert (
    convergence_full[
        "Generation"
    ].min()
    == 1
)

assert (
    convergence_full[
        "Generation"
    ].max()
    == 500
)

print(
    "\n✅ Complete convergence dataset "
    "validated."
)

In [ ]:
# ==========================================
# STEP 66: MEDIAN CONVERGENCE CURVES
# ==========================================

# Calculate median gap at every generation
# across the 30 independent runs.

median_convergence = (
    convergence_full
    .groupby(
        [
            "Instance",
            "Configuration",
            "Generation"
        ],
        observed=True
    )
    ["Optimality_Gap_Percent"]
    .median()
    .reset_index(
        name="Median_Gap_Percent"
    )
)


# ------------------------------------------
# CREATE ONE FIGURE PER INSTANCE
# ------------------------------------------

CONVERGENCE_FIGURE_DIR = os.path.join(
    RESULTS_DIR,
    "convergence_figures"
)

os.makedirs(
    CONVERGENCE_FIGURE_DIR,
    exist_ok=True
)


for instance_name in FINAL_INSTANCES:

    instance_plot = median_convergence[
        median_convergence["Instance"]
        == instance_name
    ]

    plt.figure(
        figsize=(10, 6)
    )

    for config, label in zip(
        config_order,
        config_labels
    ):

        config_plot = instance_plot[
            instance_plot["Configuration"]
            == config
        ]

        plt.plot(
            config_plot["Generation"],
            config_plot["Median_Gap_Percent"],
            linewidth=2,
            label=label
        )

    plt.xlabel(
        "Generation"
    )

    plt.ylabel(
        "Median Optimality Gap (%)"
    )

    plt.title(
        f"Convergence Behavior — {instance_name}"
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend(
        title="2-Opt Population Coverage"
    )

    plt.tight_layout()

    figure_path = os.path.join(
        CONVERGENCE_FIGURE_DIR,
        f"convergence_{instance_name}.png"
    )

    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


print(
    "Saved convergence figures to:"
)

print(
    CONVERGENCE_FIGURE_DIR
)

print(
    "\n✅ Six median convergence figures created."
)

In [ ]:
# ==========================================
# STEP 67: COMBINED CONVERGENCE FIGURE
# 2 × 3 MULTI-PANEL
# ==========================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    sharex=True
)

axes = axes.flatten()

for ax, instance_name in zip(
    axes,
    FINAL_INSTANCES
):

    instance_plot = median_convergence[
        median_convergence["Instance"]
        == instance_name
    ]

    for config, label in zip(
        config_order,
        config_labels
    ):

        config_plot = instance_plot[
            instance_plot["Configuration"]
            == config
        ]

        ax.plot(
            config_plot["Generation"],
            config_plot["Median_Gap_Percent"],
            linewidth=2,
            label=label
        )

    # Allows zero while expanding
    # the small-gap region
    ax.set_yscale(
        "symlog",
        linthresh=0.1
    )

    ax.set_title(
        instance_name
    )

    ax.grid(
        True,
        which="both",
        alpha=0.3
    )

    ax.set_xlim(
        1,
        500
    )


# ------------------------------------------
# SHARED AXIS LABELS
# ------------------------------------------

fig.supxlabel(
    "Generation",
    fontsize=12
)

fig.supylabel(
    "Median Optimality Gap (%)",
    fontsize=12
)


# ------------------------------------------
# SHARED LEGEND
# ------------------------------------------

handles, labels = (
    axes[0].get_legend_handles_labels()
)

fig.legend(
    handles,
    labels,
    title="2-Opt Population Coverage",
    loc="upper center",
    bbox_to_anchor=(0.5, 0.96),
    ncol=4
)


# ------------------------------------------
# MAIN TITLE
# ------------------------------------------

fig.suptitle(
    "Convergence Behavior Across TSPLIB Instances",
    fontsize=15,
    y=0.995
)


plt.tight_layout(
    rect=[0.03, 0.03, 1, 0.90]
)


# ------------------------------------------
# SAVE
# ------------------------------------------

COMBINED_CONVERGENCE_FILE = os.path.join(
    RESULTS_DIR,
    "figure3_combined_convergence.png"
)

plt.savefig(
    COMBINED_CONVERGENCE_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(COMBINED_CONVERGENCE_FILE)

In [ ]:
# ==========================================
# STEP 68: QUALITY–RUNTIME TRADEOFF
# ==========================================

tradeoff_data = (
    master_results
    .groupby(
        ["Instance", "Configuration"],
        observed=True
    )
    .agg(
        Mean_Runtime=(
            "Runtime_Seconds",
            "mean"
        ),
        Mean_Gap=(
            "Optimality_Gap_Percent",
            "mean"
        )
    )
    .reset_index()
)


fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9)
)

axes = axes.flatten()


for ax, instance_name in zip(
    axes,
    FINAL_INSTANCES
):

    instance_data = tradeoff_data[
        tradeoff_data["Instance"]
        == instance_name
    ]

    for config, label in zip(
        config_order,
        config_labels
    ):

        point = instance_data[
            instance_data["Configuration"]
            == config
        ]

        x = point[
            "Mean_Runtime"
        ].iloc[0]

        y = point[
            "Mean_Gap"
        ].iloc[0]

        ax.scatter(
            x,
            y,
            s=80,
            label=label
        )

        ax.annotate(
            label,
            (x, y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9
        )

    # Log scale because GA gaps
    # are much larger than hybrid gaps
    ax.set_yscale("log")

    ax.set_title(
        instance_name
    )

    ax.grid(
        True,
        which="both",
        alpha=0.3
    )


fig.supxlabel(
    "Mean Runtime (Seconds)",
    fontsize=12
)

fig.supylabel(
    "Mean Optimality Gap (%)",
    fontsize=12
)

fig.suptitle(
    "Solution Quality–Runtime Tradeoff Across TSPLIB Instances",
    fontsize=15,
    y=0.98
)

plt.tight_layout(
    rect=[0.03, 0.03, 1, 0.94]
)


FIGURE4_FILE = os.path.join(
    RESULTS_DIR,
    "figure4_quality_runtime_tradeoff.png"
)

plt.savefig(
    FIGURE4_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(FIGURE4_FILE)

In [ ]:
# ==========================================
# STEP 69: PUBLICATION RESULTS TABLE
# ==========================================

# Start from the descriptive summary
publication_table = summary_results[
    [
        "Instance",
        "Configuration",
        "Mean_Gap_Percent",
        "SD_Gap_Percent",
        "Mean_Runtime_Seconds",
        "SD_Runtime_Seconds",
        "Optimum_Reached_Percent"
    ]
].copy()


# Add convergence threshold results
publication_table = publication_table.merge(
    convergence_summary[
        [
            "Instance",
            "Configuration",
            "Reached_5pct_Percent",
            "Reached_1pct_Percent"
        ]
    ],
    on=[
        "Instance",
        "Configuration"
    ],
    how="left"
)


# ------------------------------------------
# CREATE PUBLICATION-FRIENDLY COLUMNS
# ------------------------------------------

publication_table[
    "Optimality_Gap_Mean_SD"
] = publication_table.apply(
    lambda row:
        f"{row['Mean_Gap_Percent']:.2f} ± "
        f"{row['SD_Gap_Percent']:.2f}",
    axis=1
)

publication_table[
    "Runtime_Mean_SD"
] = publication_table.apply(
    lambda row:
        f"{row['Mean_Runtime_Seconds']:.2f} ± "
        f"{row['SD_Runtime_Seconds']:.2f}",
    axis=1
)


# Rename configurations for readability
config_display_names = {
    "GA": "GA",
    "GA_2OPT_10": "GA + 2-Opt (10%)",
    "GA_2OPT_30": "GA + 2-Opt (30%)",
    "GA_2OPT_50": "GA + 2-Opt (50%)"
}

publication_table[
    "Configuration_Display"
] = publication_table[
    "Configuration"
].map(
    config_display_names
)


# ------------------------------------------
# FINAL TABLE
# ------------------------------------------

publication_table_final = publication_table[
    [
        "Instance",
        "Configuration_Display",
        "Optimality_Gap_Mean_SD",
        "Runtime_Mean_SD",
        "Reached_5pct_Percent",
        "Reached_1pct_Percent",
        "Optimum_Reached_Percent"
    ]
].copy()


publication_table_final.columns = [
    "Instance",
    "Configuration",
    "Optimality Gap (%) Mean ± SD",
    "Runtime (s) Mean ± SD",
    "Within 5% (%)",
    "Within 1% (%)",
    "Optimum Reached (%)"
]


# Round percentage columns
percentage_columns = [
    "Within 5% (%)",
    "Within 1% (%)",
    "Optimum Reached (%)"
]

publication_table_final[
    percentage_columns
] = (
    publication_table_final[
        percentage_columns
    ].round(2)
)


print(
    publication_table_final.to_string(
        index=False
    )
)


# ------------------------------------------
# SAVE
# ------------------------------------------

PUBLICATION_TABLE_FILE = os.path.join(
    RESULTS_DIR,
    "publication_results_table.csv"
)

publication_table_final.to_csv(
    PUBLICATION_TABLE_FILE,
    index=False
)

print("\nSaved to:")
print(PUBLICATION_TABLE_FILE)

assert len(
    publication_table_final
) == 24

print(
    "\n✅ Publication results table created."
)

In [ ]:
%cd /content
!git clone https://github.com/toyeentop/GA-2Opt-TSP-Intensity.git
%cd /content/GA-2Opt-TSP-Intensity